## Block 1 — Connect the new account to the shared folder


In [2]:
# Authenticate the currently signed-in Colab account, create/reuse
# Shared Root/Second Try, create a shortcut in this account's My Drive,
# and mount it as a normal filesystem path.

!pip -q install -U google-api-python-client

import time
from pathlib import Path

import google.auth
from google.colab import auth, drive
from googleapiclient.discovery import build

SHARED_ROOT_FOLDER_ID = "1b-SEdwoHhi69LwVeDHYSOTORvrsNYhi3"
SECOND_TRY_FOLDER_NAME = "Second Try"
SHORTCUT_NAME = "AMLC Second Try Shared"

auth.authenticate_user()
credentials, _ = google.auth.default()
drive_service = build("drive", "v3", credentials=credentials)


def escape_drive_query(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")


def list_drive_children(parent_id):
    rows = []
    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{parent_id}' in parents and trashed = false",
            fields=(
                "nextPageToken,files("
                "id,name,mimeType,size,parents,shortcutDetails)"
            ),
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        rows.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            return rows


root_metadata = drive_service.files().get(
    fileId=SHARED_ROOT_FOLDER_ID,
    fields="id,name,mimeType,capabilities(canAddChildren)",
    supportsAllDrives=True,
).execute()

if root_metadata.get("mimeType") != "application/vnd.google-apps.folder":
    raise RuntimeError("The supplied Drive link is not a folder.")

if not root_metadata.get("capabilities", {}).get("canAddChildren", False):
    raise PermissionError(
        "This Google account cannot edit the shared folder. "
        "Open the supplied link using this same account and confirm editor access."
    )

existing_second_try = [
    item for item in list_drive_children(SHARED_ROOT_FOLDER_ID)
    if item["name"] == SECOND_TRY_FOLDER_NAME
    and item["mimeType"] == "application/vnd.google-apps.folder"
]

if existing_second_try:
    SECOND_TRY_FOLDER_ID = existing_second_try[0]["id"]
    print("Using existing shared folder: Second Try")
else:
    created = drive_service.files().create(
        body={
            "name": SECOND_TRY_FOLDER_NAME,
            "mimeType": "application/vnd.google-apps.folder",
            "parents": [SHARED_ROOT_FOLDER_ID],
        },
        fields="id,name",
        supportsAllDrives=True,
    ).execute()
    SECOND_TRY_FOLDER_ID = created["id"]
    print("Created shared folder: Second Try")

my_drive_shortcuts = drive_service.files().list(
    q=(
        "'root' in parents and trashed = false and "
        "mimeType = 'application/vnd.google-apps.shortcut'"
    ),
    fields="files(id,name,shortcutDetails)",
    pageSize=1000,
).execute().get("files", [])

matching_shortcuts = [
    item for item in my_drive_shortcuts
    if item.get("shortcutDetails", {}).get("targetId")
    == SECOND_TRY_FOLDER_ID
]

if matching_shortcuts:
    SHORTCUT_NAME = matching_shortcuts[0]["name"]
    print("Using existing My Drive shortcut:", SHORTCUT_NAME)
else:
    shortcut = drive_service.files().create(
        body={
            "name": SHORTCUT_NAME,
            "mimeType": "application/vnd.google-apps.shortcut",
            "shortcutDetails": {"targetId": SECOND_TRY_FOLDER_ID},
            "parents": ["root"],
        },
        fields="id,name",
    ).execute()
    SHORTCUT_NAME = shortcut["name"]
    print("Created My Drive shortcut:", SHORTCUT_NAME)

drive.mount("/content/drive", force_remount=True)

SECOND_TRY_DIR = Path("/content/drive/MyDrive") / SHORTCUT_NAME

for _ in range(30):
    if SECOND_TRY_DIR.exists():
        break
    time.sleep(1)

if not SECOND_TRY_DIR.exists():
    raise FileNotFoundError(
        "The shared-folder shortcut was created but is not visible in the Drive mount. "
        "Reconnect the runtime and rerun this block."
    )

print("Shared root:", root_metadata["name"])
print("Second Try folder ID:", SECOND_TRY_FOLDER_ID)
print("Mounted Second Try path:", SECOND_TRY_DIR)


Using existing shared folder: Second Try
Using existing My Drive shortcut: AMLC Second Try Shared
Mounted at /content/drive
Shared root: amlc_sorted_neighborhood_results
Second Try folder ID: 1HsTyERTqBYX9Iu2IyU3bDs_VaIB8uVBu
Mounted Second Try path: /content/drive/MyDrive/AMLC Second Try Shared


## Block 2 — Locate and cache the competition ZIP


In [2]:
import io
import shutil

from googleapiclient.http import MediaIoBaseDownload

LOCAL_INPUT_DIR = Path("/content/amlc_second_try_input")
PERSISTENT_DATA_DIR = SECOND_TRY_DIR / "data"
LOCAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
PERSISTENT_DATA_DIR.mkdir(parents=True, exist_ok=True)

CACHED_ZIP = PERSISTENT_DATA_DIR / "student_resource.zip"
ZIP_PATH = LOCAL_INPUT_DIR / "student_resource.zip"


def recursively_find_zip(folder_id, skip_folder_id=None):
    found = []
    pending = [folder_id]
    visited = set()
    while pending:
        current = pending.pop()
        if current in visited:
            continue
        visited.add(current)
        for item in list_drive_children(current):
            if item["id"] == skip_folder_id:
                continue
            if item["mimeType"] == "application/vnd.google-apps.folder":
                pending.append(item["id"])
            elif item["name"].casefold().endswith(".zip"):
                found.append(item)
    return found


if CACHED_ZIP.exists():
    print("Restoring cached ZIP from Second Try...")
    shutil.copy2(CACHED_ZIP, ZIP_PATH)
else:
    zip_items = recursively_find_zip(
        SHARED_ROOT_FOLDER_ID,
        skip_folder_id=SECOND_TRY_FOLDER_ID,
    )

    if not zip_items:
        raise FileNotFoundError(
            "No ZIP archive was found in the supplied shared root. "
            "Upload 6ab10eb3b23ba_student_resource.zip into that shared folder, "
            "then rerun this block."
        )

    zip_items.sort(
        key=lambda item: (
            "student_resource" not in item["name"].casefold(),
            -int(item.get("size", 0) or 0),
        )
    )
    selected_zip = zip_items[0]
    print("Downloading:", selected_zip["name"])

    request = drive_service.files().get_media(fileId=selected_zip["id"])
    with open(ZIP_PATH, "wb") as stream:
        downloader = MediaIoBaseDownload(stream, request, chunksize=16 * 2**20)
        finished = False
        while not finished:
            status, finished = downloader.next_chunk()
            if status:
                print(f"Download: {100 * status.progress():5.1f}%", flush=True)

    print("Caching ZIP under Second Try/data...")
    temporary_cache = CACHED_ZIP.with_suffix(".partial")
    shutil.copy2(ZIP_PATH, temporary_cache)
    temporary_cache.replace(CACHED_ZIP)

print("ZIP:", ZIP_PATH)
print("ZIP size (GB):", round(ZIP_PATH.stat().st_size / 2**30, 3))


Restoring cached ZIP from Second Try...
ZIP: /content/amlc_second_try_input/student_resource.zip
ZIP size (GB): 1.02


## Block 3 — Extract and verify the competition data


In [3]:
WORK_DIR = Path("/content/amlc_second_try_resource")

if not WORK_DIR.exists():
    print("Extracting competition archive...")
    shutil.unpack_archive(ZIP_PATH, WORK_DIR)
else:
    print("Using existing extracted directory:", WORK_DIR)

train_matches = list(WORK_DIR.rglob("train_source1.tsv"))
test_matches = list(WORK_DIR.rglob("test_source1.tsv"))

if len(train_matches) != 1 or len(test_matches) != 1:
    raise RuntimeError(
        f"Expected one train and one test directory; found "
        f"train={len(train_matches)}, test={len(test_matches)}"
    )

TRAIN_DIR = train_matches[0].parent
TEST_DIR = test_matches[0].parent

required_train = [
    "train_source1.tsv", "train_source2.tsv", "train_source3.tsv",
    "train_ground_truth.tsv",
]
required_test = [
    "test_source1.tsv", "test_source2.tsv", "test_source3.tsv",
]

for name in required_train:
    if not (TRAIN_DIR / name).exists():
        raise FileNotFoundError(TRAIN_DIR / name)
for name in required_test:
    if not (TEST_DIR / name).exists():
        raise FileNotFoundError(TEST_DIR / name)

print("Train directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)


Extracting competition archive...
Train directory: /content/amlc_second_try_resource/student_resource/dataset/train
Test directory: /content/amlc_second_try_resource/student_resource/dataset/test


## Block 4 — Install libraries


In [4]:
# Keep Colab's preinstalled PyArrow because RAPIDS/cuDF is compiled against it.
!pip -q install -U duckdb lightgbm rapidfuzz jellyfish Unidecode
print("Libraries installed. If Colab requests a runtime restart, restart once and rerun from Block 1.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 22.3 MB/s eta 0:00:00
Libraries installed. If Colab requests a runtime restart, restart once and rerun from Block 1.


## Block 5 — Imports, configuration and persistent paths


In [5]:
import csv
import gc
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata

from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display
from tqdm.auto import tqdm
from unidecode import unidecode

SEED = 42
np.random.seed(SEED)

# Wider high-value passes plus narrower complementary passes.
SORT_PASSES = {
    "name": {"column": "sn_key_name", "radius": 20},
    "name_tokens": {"column": "sn_key_name_tokens", "radius": 20},
    "address_numbers": {"column": "sn_key_address_numbers", "radius": 12},
    "address_tokens": {"column": "sn_key_address_tokens", "radius": 15},
    "name_core": {"column": "sn_key_name_core", "radius": 20},
    "name_acronym": {"column": "sn_key_name_acronym", "radius": 8},
    "postal_house": {"column": "sn_key_postal_house", "radius": 12},
    "address_tail": {"column": "sn_key_address_tail", "radius": 12},
}

WINDOW_RADIUS = max(item["radius"] for item in SORT_PASSES.values())
VALIDATION_MODULUS = 5
VALIDATION_BUCKET = 0

# Previous run used 5. A value of 2 emits validation S1 plus half of
# non-validation S1 for model training.
MODEL_TRAIN_QUERY_SAMPLE_MODULUS = 2

CANDIDATE_BUCKETS = 32
FEATURE_PARTITIONS = CANDIDATE_BUCKETS
MAX_TRAIN_NEGATIVES = 3_000_000
MAX_HARD_NEGATIVES = 2_000_000
HARD_NEGATIVE_MIN_PROBABILITY = 0.20
BASELINE_TREES = 150
FINAL_TREES = 450
NEGATIVE_PAIR_WEIGHT = 2.0
HARD_NEGATIVE_WEIGHT = 3.0
NORMALIZE_CHUNK_SIZE = 150_000

LOCAL_ROOT = Path("/content/amlc_second_try")
LOCAL_TRAIN = LOCAL_ROOT / "train"
LOCAL_TEST = LOCAL_ROOT / "test"
LOCAL_TEMP = LOCAL_ROOT / "duckdb_temp"

PERSIST_ROOT = SECOND_TRY_DIR
PERSIST_TRAIN = PERSIST_ROOT / "training"
PERSIST_TEST = PERSIST_ROOT / "testing"
PERSIST_MODELS = PERSIST_ROOT / "models"
PERSIST_OUTPUTS = PERSIST_ROOT / "outputs"

for directory in [
    LOCAL_ROOT, LOCAL_TRAIN, LOCAL_TEST, LOCAL_TEMP,
    PERSIST_TRAIN, PERSIST_TEST, PERSIST_MODELS, PERSIST_OUTPUTS,
]:
    directory.mkdir(parents=True, exist_ok=True)


def atomic_copy(source, destination):
    source = Path(source)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".partial")
    if temporary.exists():
        temporary.unlink()
    started = time.perf_counter()
    shutil.copy2(source, temporary)
    temporary.replace(destination)
    print(
        "Checkpointed:", destination,
        "| size GB:", round(destination.stat().st_size / 2**30, 3),
        "| minutes:", round((time.perf_counter() - started) / 60, 2),
    )


def restore_file(persistent, local):
    persistent = Path(persistent)
    local = Path(local)
    if local.exists():
        return True
    if not persistent.exists():
        return False
    local.parent.mkdir(parents=True, exist_ok=True)
    print("Restoring:", persistent.name)
    shutil.copy2(persistent, local)
    return True


def create_connection(temp_directory=LOCAL_TEMP, memory_limit="6GB", threads=4):
    temp_directory = Path(temp_directory)
    temp_directory.mkdir(parents=True, exist_ok=True)
    connection = duckdb.connect()
    connection.execute("SET preserve_insertion_order = false")
    connection.execute(f"SET threads = {max(1, min(threads, os.cpu_count() or 1))}")
    connection.execute(f"SET memory_limit = '{memory_limit}'")
    connection.execute(f"SET temp_directory = '{temp_directory.as_posix()}'")
    connection.execute("SET enable_progress_bar = true")
    connection.execute("SET progress_bar_time = 1000")
    return connection


TRAIN_NORMALIZED_LOCAL = LOCAL_TRAIN / "normalized"
TRAIN_NORMALIZED_DRIVE = PERSIST_TRAIN / "normalized"
TEST_NORMALIZED_LOCAL = LOCAL_TEST / "normalized"
TEST_NORMALIZED_DRIVE = PERSIST_TEST / "normalized"

for directory in [
    TRAIN_NORMALIZED_LOCAL, TRAIN_NORMALIZED_DRIVE,
    TEST_NORMALIZED_LOCAL, TEST_NORMALIZED_DRIVE,
]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_NORMALIZED_FILES = {
    "s1": TRAIN_NORMALIZED_LOCAL / "train_s1.parquet",
    "s2": TRAIN_NORMALIZED_LOCAL / "train_s2.parquet",
    "s3": TRAIN_NORMALIZED_LOCAL / "train_s3.parquet",
}
TEST_NORMALIZED_FILES = {
    "s1": TEST_NORMALIZED_LOCAL / "test_s1.parquet",
    "s2": TEST_NORMALIZED_LOCAL / "test_s2.parquet",
    "s3": TEST_NORMALIZED_LOCAL / "test_s3.parquet",
}

print("Second Try persistent root:", PERSIST_ROOT)
print("Passes:", {k: v["radius"] for k, v in SORT_PASSES.items()})
print("Training sample modulus:", MODEL_TRAIN_QUERY_SAMPLE_MODULUS)
print("Candidate/feature shards:", CANDIDATE_BUCKETS)
print("Free local disk GB:", round(shutil.disk_usage("/content").free / 2**30, 2))


Second Try persistent root: /content/drive/MyDrive/AMLC Second Try Shared
Passes: {'name': 20, 'name_tokens': 20, 'address_numbers': 12, 'address_tokens': 15, 'name_core': 20, 'name_acronym': 8, 'postal_house': 12, 'address_tail': 12}
Training sample modulus: 2
Candidate/feature shards: 32
Free local disk GB: 82.84


## Block 6 — Enhanced normalization and sorting keys


In [6]:
NON_ALPHANUMERIC_PATTERN = re.compile(r"[^a-z0-9]+")
NUMBER_PATTERN = re.compile(r"\d+")
POSTAL_PATTERN = re.compile(r"^\d{5,6}$")

LEGAL_SUFFIX_TOKENS = {
    "inc", "incorporated", "corp", "corporation", "company", "co",
    "llc", "llp", "ltd", "limited", "pvt", "private", "plc",
    "gmbh", "ag", "sa", "sas", "sarl", "bv", "nv", "oy", "ab",
    "spa", "srl", "pte", "enterprises", "services",
}


def normalize_text(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = unidecode(text).casefold().replace("&", " and ")
    text = NON_ALPHANUMERIC_PATTERN.sub(" ", text)
    return " ".join(text.split())


def sorted_unique_tokens(text):
    return " ".join(sorted(set(text.split()))) if text else ""


def numeric_signature(text):
    return " ".join(sorted(set(NUMBER_PATTERN.findall(text)))) if text else ""


def remove_legal_suffixes(text):
    tokens = text.split()
    while len(tokens) > 1 and tokens[-1] in LEGAL_SUFFIX_TOKENS:
        tokens.pop()
    return " ".join(tokens)


def acronym_signature(text):
    tokens = [t for t in text.split() if t not in LEGAL_SUFFIX_TOKENS]
    if len(tokens) < 2:
        return ""
    return "".join(token[0] for token in tokens if token)


def first_number(text):
    match = NUMBER_PATTERN.search(text or "")
    return match.group(0) if match else ""


def postal_signature(text):
    matches = [token for token in (text or "").split() if POSTAL_PATTERN.fullmatch(token)]
    return matches[-1] if matches else ""


def address_tail_signature(text):
    tokens = [token for token in (text or "").split() if not token.isdigit()]
    return " ".join(tokens[-3:])


def normalize_chunk(frame):
    frame = frame.copy()
    frame["entity_id"] = frame["entity_id"].fillna("").astype(str)
    frame["country_norm"] = frame["country"].fillna("").astype(str).map(normalize_text)
    frame["name_norm"] = frame["business_name"].fillna("").astype(str).map(normalize_text)
    frame["address_norm"] = frame["business_address"].fillna("").astype(str).map(normalize_text)

    frame["name_core"] = frame["name_norm"].map(remove_legal_suffixes)
    frame["name_acronym"] = frame["name_core"].map(acronym_signature)
    frame["name_tokens_sorted"] = frame["name_core"].map(sorted_unique_tokens)
    frame["address_tokens_sorted"] = frame["address_norm"].map(sorted_unique_tokens)
    frame["address_numbers"] = frame["address_norm"].map(numeric_signature)
    frame["house_number"] = frame["address_norm"].map(first_number)
    frame["postal_code"] = frame["address_norm"].map(postal_signature)
    frame["address_tail"] = frame["address_norm"].map(address_tail_signature)

    def combine(*columns):
        result = frame[columns[0]].fillna("")
        for column in columns[1:]:
            result = result + " | " + frame[column].fillna("")
        return result.str.strip(" |")

    frame["sn_key_name"] = combine("name_norm", "postal_code", "house_number")
    frame["sn_key_name_tokens"] = combine("name_tokens_sorted", "postal_code", "house_number")
    frame["sn_key_address_numbers"] = combine("address_numbers", "address_tail", "name_core")
    frame["sn_key_address_tokens"] = combine("address_tokens_sorted", "name_core")
    frame["sn_key_name_core"] = combine("name_core", "postal_code", "address_tail")
    frame["sn_key_name_acronym"] = combine("name_acronym", "postal_code", "house_number")
    frame["sn_key_postal_house"] = combine("postal_code", "house_number", "name_core")
    frame["sn_key_address_tail"] = combine("address_tail", "house_number", "name_core")

    output_columns = [
        "entity_id", "country_norm", "name_norm", "name_core", "name_acronym",
        "address_norm", "address_numbers", "house_number", "postal_code", "address_tail",
    ] + [item["column"] for item in SORT_PASSES.values()]

    return frame[list(dict.fromkeys(output_columns))]


print("Enhanced normalization functions ready.")


Enhanced normalization functions ready.


## Block 7 — Normalize train data with Drive checkpoints


In [7]:
def normalize_tsv_to_parquet(input_path, local_output, persistent_output, label):
    if restore_file(persistent_output, local_output):
        print(label, "restored/available:", local_output)
        return

    temporary = local_output.with_suffix(".building.parquet")
    if temporary.exists():
        temporary.unlink()

    writer = None
    total_rows = 0
    started = time.perf_counter()

    reader = pd.read_csv(
        input_path,
        sep="\t",
        chunksize=NORMALIZE_CHUNK_SIZE,
        keep_default_na=False,
    )

    for chunk_number, chunk in enumerate(reader, start=1):
        normalized = normalize_chunk(chunk)
        table = pa.Table.from_pandas(normalized, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(temporary, table.schema, compression="zstd")
        writer.write_table(table, row_group_size=100_000)
        total_rows += len(normalized)
        elapsed = time.perf_counter() - started
        print(
            f"{label}: chunk={chunk_number:,} rows={total_rows:,} "
            f"elapsed={elapsed / 60:.1f} min",
            flush=True,
        )
        del chunk, normalized, table
        gc.collect()

    if writer is None:
        raise RuntimeError(f"No rows read for {label}")
    writer.close()
    temporary.replace(local_output)
    atomic_copy(local_output, persistent_output)


train_inputs = {
    "s1": TRAIN_DIR / "train_source1.tsv",
    "s2": TRAIN_DIR / "train_source2.tsv",
    "s3": TRAIN_DIR / "train_source3.tsv",
}

for source in ["s1", "s2", "s3"]:
    normalize_tsv_to_parquet(
        train_inputs[source],
        TRAIN_NORMALIZED_FILES[source],
        TRAIN_NORMALIZED_DRIVE / TRAIN_NORMALIZED_FILES[source].name,
        f"train {source.upper()}",
    )

print("All training sources normalized.")


train S1: chunk=1 rows=150,000 elapsed=0.2 min
train S1: chunk=2 rows=300,000 elapsed=0.3 min
train S1: chunk=3 rows=450,000 elapsed=0.5 min
train S1: chunk=4 rows=600,000 elapsed=0.5 min
train S1: chunk=5 rows=750,000 elapsed=0.6 min
train S1: chunk=6 rows=900,000 elapsed=0.7 min
train S1: chunk=7 rows=1,050,000 elapsed=0.8 min
train S1: chunk=8 rows=1,200,000 elapsed=0.9 min
train S1: chunk=9 rows=1,350,000 elapsed=1.0 min
train S1: chunk=10 rows=1,500,000 elapsed=1.1 min
train S1: chunk=11 rows=1,650,000 elapsed=1.2 min
train S1: chunk=12 rows=1,800,000 elapsed=1.3 min
train S1: chunk=13 rows=1,950,000 elapsed=1.4 min
train S1: chunk=14 rows=2,100,000 elapsed=1.5 min
train S1: chunk=15 rows=2,206,821 elapsed=1.6 min
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/normalized/train_s1.parquet | size GB: 0.43 | minutes: 0.03
train S2: chunk=1 rows=150,000 elapsed=0.1 min
train S2: chunk=2 rows=300,000 elapsed=0.3 min
train S2: chunk=3 rows=450,000 elapsed=0.4 min
t

## Block 8 — Ground truth and deterministic S1 split flags


In [8]:
TRUTH_PAIRS_FILE = LOCAL_TRAIN / "truth_pairs.parquet"
TRUTH_PAIRS_DRIVE = PERSIST_TRAIN / "truth_pairs.parquet"
TRAIN_S1_FLAGGED_FILE = LOCAL_TRAIN / "train_s1_flagged.parquet"
TRAIN_S1_FLAGGED_DRIVE = PERSIST_TRAIN / "train_s1_flagged.parquet"


def build_truth_pairs():
    if restore_file(TRUTH_PAIRS_DRIVE, TRUTH_PAIRS_FILE):
        return
    truth = pd.read_csv(
        TRAIN_DIR / "train_ground_truth.tsv",
        sep="\t",
        keep_default_na=False,
    )
    rows = []
    for row in tqdm(truth.itertuples(index=False), total=len(truth), desc="Expanding truth"):
        source1_id = str(row.source1_entity_id).strip()
        for candidate_id in str(row.matched_entity_ids or "").split(","):
            candidate_id = candidate_id.strip()
            if candidate_id:
                rows.append((source1_id, candidate_id))
    table = pa.Table.from_pydict({
        "source1_entity_id": [row[0] for row in rows],
        "candidate_entity_id": [row[1] for row in rows],
    })
    pq.write_table(table, TRUTH_PAIRS_FILE, compression="zstd")
    atomic_copy(TRUTH_PAIRS_FILE, TRUTH_PAIRS_DRIVE)
    print("Truth links:", f"{len(rows):,}")


build_truth_pairs()

if not restore_file(TRAIN_S1_FLAGGED_DRIVE, TRAIN_S1_FLAGGED_FILE):
    connection = create_connection()
    connection.execute(f"""
        COPY (
            SELECT
                *,
                CAST(
                    MOD(hash(entity_id), {VALIDATION_MODULUS}) = {VALIDATION_BUCKET}
                    AS TINYINT
                ) AS split_code,
                CAST(
                    MOD(hash(entity_id), {VALIDATION_MODULUS}) = {VALIDATION_BUCKET}
                    OR MOD(hash(entity_id || '|enhanced_train'),
                           {MODEL_TRAIN_QUERY_SAMPLE_MODULUS}) = 0
                    AS BOOLEAN
                ) AS emit_query
            FROM read_parquet('{TRAIN_NORMALIZED_FILES["s1"].as_posix()}')
        ) TO '{TRAIN_S1_FLAGGED_FILE.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 150000)
    """)
    connection.close()
    atomic_copy(TRAIN_S1_FLAGGED_FILE, TRAIN_S1_FLAGGED_DRIVE)

connection = create_connection()
split_summary = connection.execute(f"""
    SELECT split_code, emit_query, COUNT(*) AS rows
    FROM read_parquet('{TRAIN_S1_FLAGGED_FILE.as_posix()}')
    GROUP BY split_code, emit_query
    ORDER BY split_code, emit_query
""").df()
connection.close()
display(split_summary)


Expanding truth:   0%|          | 0/2206821 [00:00<?, ?it/s]

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/truth_pairs.parquet | size GB: 0.056 | minutes: 0.0
Truth links: 7,638,365


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/train_s1_flagged.parquet | size GB: 0.36 | minutes: 0.03


,split_code,emit_query,rows
0,0,False,882244
1,0,True,883483
2,1,True,441094


## Block 9 — GPU setup and resumable Sorted-Neighborhood functions


In [ ]:
import cupy as cp
import cudf

if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("No CUDA GPU detected. Select a Colab GPU runtime.")


def gpu_memory_status():
    free_bytes, total_bytes = cp.cuda.runtime.memGetInfo()
    return {
        "used_GB": round((total_bytes - free_bytes) / 2**30, 2),
        "free_GB": round(free_bytes / 2**30, 2),
        "total_GB": round(total_bytes / 2**30, 2),
    }


gpu_name = cp.cuda.runtime.getDeviceProperties(0)["name"]
if isinstance(gpu_name, bytes):
    gpu_name = gpu_name.decode()
print("GPU:", gpu_name)
print("GPU memory:", gpu_memory_status())

PASS_CODES = {name: code for code, name in enumerate(SORT_PASSES)}


def load_gpu_source(path, key_column, country, source_code, is_s1=False):
    columns = ["entity_id", "country_norm", key_column]
    if is_s1:
        columns += ["split_code", "emit_query"]
    frame = cudf.read_parquet(
        str(path),
        columns=columns,
        filters=[("country_norm", "==", country)],
    ).rename(columns={key_column: "sort_key"})
    frame = frame[
        frame["sort_key"].notna()
        & (frame["sort_key"].str.len() >= 2)
    ]
    result = cudf.DataFrame()
    result["entity_id"] = frame["entity_id"]
    result["sort_key"] = frame["sort_key"]
    result["source_code"] = cp.int8(source_code)
    if is_s1:
        result["split_code"] = frame["split_code"].astype("int8")
        result["emit_query"] = frame["emit_query"].astype("bool")
    else:
        result["split_code"] = cp.int8(-1)
        result["emit_query"] = False
    del frame
    return result.reset_index(drop=True)


def collect_gpu_direction(ordered, distance, shift_amount, pass_code):
    neighbour_id = ordered["entity_id"].shift(shift_amount)
    neighbour_source = ordered["source_code"].shift(shift_amount)
    mask = ordered["emit_query"] & neighbour_source.notna() & (neighbour_source > 0)
    if int(mask.sum()) == 0:
        return None
    pairs = cudf.DataFrame()
    pairs["source1_entity_id"] = ordered.loc[mask, "entity_id"].reset_index(drop=True)
    pairs["candidate_entity_id"] = neighbour_id.loc[mask].reset_index(drop=True)
    pairs["candidate_source_code"] = neighbour_source.loc[mask].astype("int8").reset_index(drop=True)
    pairs["pass_code"] = cp.int8(pass_code)
    pairs["sn_rank_distance"] = cp.int16(distance)
    pairs["split_code"] = ordered.loc[mask, "split_code"].astype("int8").reset_index(drop=True)
    return pairs


def generate_gpu_pass_for_country(
    label, pass_name, country,
    s1_path, s2_path, s3_path,
    local_parts_dir, persistent_parts_dir,
):
    key_column = SORT_PASSES[pass_name]["column"]
    radius = SORT_PASSES[pass_name]["radius"]
    pass_code = PASS_CODES[pass_name]
    local_parts_dir.mkdir(parents=True, exist_ok=True)
    persistent_parts_dir.mkdir(parents=True, exist_ok=True)
    country_slug = re.sub(r"[^a-z0-9]+", "_", country.casefold()).strip("_")

    print("\n" + "=" * 88)
    print(f"{label.upper()} | pass={pass_name} | country={country} | radius={radius}")
    print("GPU before load:", gpu_memory_status())
    job_started = time.perf_counter()

    s1 = load_gpu_source(s1_path, key_column, country, 0, True)
    s2 = load_gpu_source(s2_path, key_column, country, 2, False)
    s3 = load_gpu_source(s3_path, key_column, country, 3, False)
    print(f"Loaded S1={len(s1):,}, S2={len(s2):,}, S3={len(s3):,}")

    ordered = cudf.concat([s1, s2, s3], ignore_index=True)
    del s1, s2, s3
    sort_started = time.perf_counter()
    ordered = ordered.sort_values(["sort_key", "entity_id"], ignore_index=True)
    ordered = ordered.drop(columns=["sort_key"])
    print(f"Sorted {len(ordered):,} rows in {(time.perf_counter()-sort_started)/60:.2f} min")

    total_pairs = 0
    for distance in range(1, radius + 1):
        filename = (
            f"{label}_gpu_{pass_code:02d}_{pass_name}_{country_slug}_"
            f"d{distance:02d}.parquet"
        )
        local_path = local_parts_dir / filename
        drive_path = persistent_parts_dir / filename

        if local_path.exists():
            print(f"distance {distance:02d}/{radius}: local checkpoint exists")
            continue
        if drive_path.exists():
            shutil.copy2(drive_path, local_path)
            print(f"distance {distance:02d}/{radius}: restored from Drive")
            continue

        started = time.perf_counter()
        below = collect_gpu_direction(ordered, distance, -distance, pass_code)
        above = collect_gpu_direction(ordered, distance, distance, pass_code)
        pieces = [piece for piece in (below, above) if piece is not None]
        pair_count = 0
        if pieces:
            pairs = cudf.concat(pieces, ignore_index=True)
            pair_count = len(pairs)
            pairs.to_parquet(str(local_path), compression="snappy", index=False)
            atomic_copy(local_path, drive_path)
            del pairs
        total_pairs += pair_count
        del below, above, pieces
        gc.collect()
        cp.get_default_memory_pool().free_all_blocks()
        print(
            f"distance {distance:02d}/{radius}: pairs={pair_count:,} "
            f"cumulative={total_pairs:,} seconds={time.perf_counter()-started:.1f}"
        )

    del ordered
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    print(
        f"Finished {label}/{pass_name}/{country} in "
        f"{(time.perf_counter()-job_started)/60:.2f} min | GPU {gpu_memory_status()}"
    )


print("Resumable GPU candidate functions ready.")


## Block 10 — Generate enhanced training candidates


In [10]:
TRAIN_CANDIDATE_PARTS_LOCAL = LOCAL_TRAIN / "candidate_distance_parts"
TRAIN_CANDIDATE_PARTS_DRIVE = PERSIST_TRAIN / "candidate_distance_parts"
TRAIN_CANDIDATE_PARTS_LOCAL.mkdir(parents=True, exist_ok=True)
TRAIN_CANDIDATE_PARTS_DRIVE.mkdir(parents=True, exist_ok=True)

connection = create_connection()
TRAIN_COUNTRIES = [
    row[0] for row in connection.execute(f"""
        SELECT DISTINCT country_norm
        FROM read_parquet('{TRAIN_S1_FLAGGED_FILE.as_posix()}')
        WHERE country_norm <> ''
        ORDER BY country_norm
    """).fetchall()
]
connection.close()

total_jobs = len(SORT_PASSES) * len(TRAIN_COUNTRIES)
job = 0
started = time.perf_counter()
for pass_name in SORT_PASSES:
    for country in TRAIN_COUNTRIES:
        job += 1
        print(f"\nTRAIN JOB {job}/{total_jobs}")
        generate_gpu_pass_for_country(
            label="train",
            pass_name=pass_name,
            country=country,
            s1_path=TRAIN_S1_FLAGGED_FILE,
            s2_path=TRAIN_NORMALIZED_FILES["s2"],
            s3_path=TRAIN_NORMALIZED_FILES["s3"],
            local_parts_dir=TRAIN_CANDIDATE_PARTS_LOCAL,
            persistent_parts_dir=TRAIN_CANDIDATE_PARTS_DRIVE,
        )

print("Training candidate generation minutes:", round((time.perf_counter()-started)/60, 2))
print("Distance part files:", len(list(TRAIN_CANDIDATE_PARTS_LOCAL.glob("train_gpu_*.parquet"))))



TRAIN JOB 1/16

TRAIN | pass=name | country=india | radius=20
GPU before load: {'used_GB': 0.1, 'free_GB': 14.46, 'total_GB': 14.56}
Loaded S1=883,188, S2=2,017,799, S3=2,115,547
Sorted 5,016,534 rows in 0.01 min
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/candidate_distance_parts/train_gpu_00_name_india_d01.parquet | size GB: 0.014 | minutes: 0.0
distance 01/20: pairs=780,861 cumulative=780,861 seconds=0.9
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/candidate_distance_parts/train_gpu_00_name_india_d02.parquet | size GB: 0.014 | minutes: 0.0
distance 02/20: pairs=747,339 cumulative=1,528,200 seconds=0.6
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/candidate_distance_parts/train_gpu_00_name_india_d03.parquet | size GB: 0.014 | minutes: 0.0
distance 03/20: pairs=728,386 cumulative=2,256,586 seconds=0.5
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/candidate_distance_parts/train_gpu_00_name_i

## Block 11 — Bounded-memory union into 32 candidate shards


In [11]:
def union_candidate_parts(
    label,
    parts_directory,
    part_pattern,
    local_shard_directory,
    persistent_shard_directory,
    temporary_directory,
):
    local_shard_directory.mkdir(parents=True, exist_ok=True)
    persistent_shard_directory.mkdir(parents=True, exist_ok=True)
    temporary_directory.mkdir(parents=True, exist_ok=True)

    expected_local = [
        local_shard_directory / f"{label}_candidates_{bucket:03d}.parquet"
        for bucket in range(CANDIDATE_BUCKETS)
    ]
    expected_drive = [
        persistent_shard_directory / path.name for path in expected_local
    ]

    for drive_path, local_path in zip(expected_drive, expected_local):
        restore_file(drive_path, local_path)

    if all(path.exists() for path in expected_local):
        print(label, "candidate shards already complete.")
        return local_shard_directory / f"{label}_candidates_*.parquet"

    part_files = sorted(parts_directory.glob(part_pattern))
    if not part_files:
        raise RuntimeError(f"No candidate distance parts found for {label}.")

    stage_dir = temporary_directory / f"{label}_bucketed_raw"
    if stage_dir.exists():
        shutil.rmtree(stage_dir)

    connection = create_connection(temporary_directory, memory_limit="4GB", threads=2)
    wildcard = (parts_directory / part_pattern).as_posix()
    print(f"Partitioning {len(part_files):,} {label} distance files into {CANDIDATE_BUCKETS} buckets...")
    stage_started = time.perf_counter()
    connection.execute(f"""
        COPY (
            SELECT *, MOD(hash(source1_entity_id), {CANDIDATE_BUCKETS}) AS candidate_bucket
            FROM read_parquet('{wildcard}')
        ) TO '{stage_dir.as_posix()}'
        (FORMAT PARQUET, PARTITION_BY(candidate_bucket), COMPRESSION ZSTD)
    """)
    print("Raw partition minutes:", round((time.perf_counter()-stage_started)/60, 2))

    flag_sql = ",\n".join(
        f"MAX(CAST(pass_code = {PASS_CODES[name]} AS TINYINT)) AS sn_from_{name}"
        for name in SORT_PASSES
    )

    progress = tqdm(range(CANDIDATE_BUCKETS), desc=f"Union {label} buckets")
    for bucket in progress:
        local_output = expected_local[bucket]
        drive_output = expected_drive[bucket]
        if local_output.exists():
            continue
        bucket_pattern = stage_dir / f"candidate_bucket={bucket}" / "*.parquet"
        connection.execute(f"""
            COPY (
                SELECT
                    source1_entity_id,
                    candidate_entity_id,
                    CASE MAX(candidate_source_code) WHEN 3 THEN 'S3' ELSE 'S2' END AS candidate_source,
                    CASE MAX(split_code) WHEN 1 THEN 'validation' ELSE 'model_train' END AS split,
                    MIN(sn_rank_distance) AS sn_min_rank_distance,
                    COUNT(DISTINCT pass_code) AS sn_pass_count,
                    {flag_sql}
                FROM read_parquet('{bucket_pattern.as_posix()}')
                GROUP BY source1_entity_id, candidate_entity_id
            ) TO '{local_output.as_posix()}'
            (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
        """)
        atomic_copy(local_output, drive_output)
        progress.set_postfix(file=local_output.name)

    connection.close()
    shutil.rmtree(stage_dir)

    if not all(path.exists() for path in expected_local):
        raise RuntimeError(f"Incomplete {label} candidate shards.")
    return local_shard_directory / f"{label}_candidates_*.parquet"


TRAIN_CANDIDATE_SHARDS_LOCAL = LOCAL_TRAIN / "candidate_shards"
TRAIN_CANDIDATE_SHARDS_DRIVE = PERSIST_TRAIN / "candidate_shards"

TRAIN_CANDIDATE_PATTERN = union_candidate_parts(
    label="train",
    parts_directory=TRAIN_CANDIDATE_PARTS_LOCAL,
    part_pattern="train_gpu_*.parquet",
    local_shard_directory=TRAIN_CANDIDATE_SHARDS_LOCAL,
    persistent_shard_directory=TRAIN_CANDIDATE_SHARDS_DRIVE,
    temporary_directory=LOCAL_TEMP / "train_union",
)

connection = create_connection()
train_candidate_count = connection.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{TRAIN_CANDIDATE_PATTERN.as_posix()}')
""").fetchone()[0]
candidate_summary = connection.execute(f"""
    WITH counts AS (
        SELECT source1_entity_id, split, COUNT(*) AS candidate_count
        FROM read_parquet('{TRAIN_CANDIDATE_PATTERN.as_posix()}')
        GROUP BY source1_entity_id, split
    )
    SELECT split, COUNT(*) AS s1_with_candidates,
           AVG(candidate_count) AS average_candidates,
           MEDIAN(candidate_count) AS median_candidates,
           QUANTILE_CONT(candidate_count, 0.95) AS p95_candidates,
           MAX(candidate_count) AS maximum_candidates
    FROM counts GROUP BY split ORDER BY split
""").df()
connection.close()
print("Training candidate pairs:", f"{train_candidate_count:,}")
display(candidate_summary)


Partitioning 238 train distance files into 32 buckets...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw partition minutes: 2.76


Union train buckets:   0%|          | 0/32 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/candidate_shards/train_candidates_031.parquet | size GB: 0.054 | minutes: 0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training candidate pairs: 172,564,964


,split,s1_with_candidates,average_candidates,median_candidates,p95_candidates,maximum_candidates
0,model_train,883483,130.293672,131.0,154.0,202
1,validation,441094,130.250513,131.0,154.0,196


## Block 12 — Candidate recall and oracle validation ceiling


In [12]:
connection = create_connection(memory_limit="5GB", threads=2)
candidate_metrics = connection.execute(f"""
    WITH validation_entities AS (
        SELECT entity_id AS source1_entity_id
        FROM read_parquet('{TRAIN_S1_FLAGGED_FILE.as_posix()}')
        WHERE split_code = 1
    ),
    truth_counts AS (
        SELECT v.source1_entity_id,
               COUNT(DISTINCT t.candidate_entity_id) AS truth_count
        FROM validation_entities v
        LEFT JOIN read_parquet('{TRUTH_PAIRS_FILE.as_posix()}') t
        USING(source1_entity_id)
        GROUP BY v.source1_entity_id
    ),
    retrieved AS (
        SELECT c.source1_entity_id,
               COUNT(DISTINCT c.candidate_entity_id) AS retrieved_count
        FROM read_parquet('{TRAIN_CANDIDATE_PATTERN.as_posix()}') c
        INNER JOIN read_parquet('{TRUTH_PAIRS_FILE.as_posix()}') t
          ON c.source1_entity_id = t.source1_entity_id
         AND c.candidate_entity_id = t.candidate_entity_id
        WHERE c.split = 'validation'
        GROUP BY c.source1_entity_id
    ),
    per_entity AS (
        SELECT tc.source1_entity_id, tc.truth_count,
               COALESCE(r.retrieved_count, 0) AS retrieved_count,
               CASE
                 WHEN tc.truth_count = 0 THEN 1.0
                 WHEN COALESCE(r.retrieved_count, 0) = 0 THEN 0.0
                 ELSE 1.25 * r.retrieved_count /
                      (1.25 * r.retrieved_count + 0.25 * (tc.truth_count-r.retrieved_count))
               END AS oracle_f0_5
        FROM truth_counts tc LEFT JOIN retrieved r USING(source1_entity_id)
    )
    SELECT
        COUNT(*) AS validation_s1,
        SUM(CAST(truth_count = 0 AS BIGINT)) AS singletons,
        SUM(truth_count) AS true_links,
        SUM(retrieved_count) AS retrieved_true_links,
        SUM(retrieved_count)::DOUBLE / NULLIF(SUM(truth_count), 0) AS candidate_link_recall,
        AVG(CASE WHEN truth_count > 0 THEN CAST(retrieved_count > 0 AS DOUBLE) END) AS at_least_one_coverage,
        AVG(CASE WHEN truth_count > 0 THEN CAST(retrieved_count = truth_count AS DOUBLE) END) AS complete_match_coverage,
        AVG(oracle_f0_5) AS oracle_macro_f0_5
    FROM per_entity
""").df()
connection.close()
display(candidate_metrics.T)
recall = float(candidate_metrics.iloc[0]["candidate_link_recall"])
oracle = float(candidate_metrics.iloc[0]["oracle_macro_f0_5"])
print("Candidate recall target >= 0.92 | actual:", recall)
print("Oracle macro F0.5 target >= 0.96 | actual:", oracle)
if recall < 0.90:
    print("WARNING: candidate recall remains below 0.90; a 0.95 final score is unlikely.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,0
validation_s1,4.410940e+05
singletons,2.461800e+04
true_links,1.527077e+06
retrieved_true_links,1.375169e+06
candidate_link_recall,9.005237e-01
at_least_one_coverage,9.857111e-01
complete_match_coverage,7.409166e-01
oracle_macro_f0_5,9.607998e-01


Candidate recall target >= 0.92 | actual: 0.9005236802073504
Oracle macro F0.5 target >= 0.96 | actual: 0.9607997772553555


## Block 13 — Build resumable pair-feature shards


In [13]:
PASS_FEATURE_COLUMNS = [f"sn_from_{name}" for name in SORT_PASSES]


def token_jaccard_sql(left_expression, right_expression):
    return f"""
        CASE
            WHEN {left_expression} = '' OR {right_expression} = '' THEN 0.0
            ELSE list_count(list_intersect(
                     list_distinct(string_split({left_expression}, ' ')),
                     list_distinct(string_split({right_expression}, ' '))
                 ))::DOUBLE /
                 NULLIF(list_count(list_distinct(list_concat(
                     string_split({left_expression}, ' '),
                     string_split({right_expression}, ' ')
                 ))), 0)
        END
    """


def similarity_feature_sql(include_label):
    name_jaccard = token_jaccard_sql("s1.name_core", "target.name_core")
    address_jaccard = token_jaccard_sql("s1.address_norm", "target.address_norm")
    number_jaccard = token_jaccard_sql("s1.address_numbers", "target.address_numbers")
    pass_columns = ",\n".join(
        f"CAST(candidates.{column} AS FLOAT) AS {column}"
        for column in PASS_FEATURE_COLUMNS
    )
    label_sql = (
        "CAST(truth.candidate_entity_id IS NOT NULL AS INTEGER) AS label"
        if include_label else "CAST(0 AS INTEGER) AS label"
    )
    return f"""
        candidates.source1_entity_id,
        candidates.candidate_entity_id,
        candidates.split,
        CAST(candidates.sn_min_rank_distance AS FLOAT) AS sn_min_rank_distance,
        CAST(candidates.sn_pass_count AS FLOAT) AS sn_pass_count,
        {pass_columns},
        CAST(candidates.candidate_source = 'S3' AS FLOAT) AS candidate_source_is_s3,

        CAST(s1.name_norm <> '' AND s1.name_norm = target.name_norm AS FLOAT) AS name_exact,
        CAST(s1.name_core <> '' AND s1.name_core = target.name_core AS FLOAT) AS name_core_exact,
        CAST(s1.name_acronym <> '' AND s1.name_acronym = target.name_acronym AS FLOAT) AS name_acronym_exact,
        CAST(jaro_winkler_similarity(s1.name_norm, target.name_norm) AS FLOAT) AS name_jaro_winkler,
        CAST(jaro_winkler_similarity(s1.name_core, target.name_core) AS FLOAT) AS name_core_jaro_winkler,
        CAST(1.0 - levenshtein(s1.name_norm, target.name_norm)::DOUBLE /
             GREATEST(LENGTH(s1.name_norm), LENGTH(target.name_norm), 1) AS FLOAT)
             AS name_levenshtein_ratio,
        CAST(({name_jaccard}) AS FLOAT) AS name_token_jaccard,
        CAST(LEAST(LENGTH(s1.name_norm), LENGTH(target.name_norm))::DOUBLE /
             GREATEST(LENGTH(s1.name_norm), LENGTH(target.name_norm), 1) AS FLOAT)
             AS name_length_ratio,

        CAST(s1.address_norm <> '' AND s1.address_norm = target.address_norm AS FLOAT) AS address_exact,
        CAST(jaro_winkler_similarity(s1.address_norm, target.address_norm) AS FLOAT) AS address_jaro_winkler,
        CAST(1.0 - levenshtein(s1.address_norm, target.address_norm)::DOUBLE /
             GREATEST(LENGTH(s1.address_norm), LENGTH(target.address_norm), 1) AS FLOAT)
             AS address_levenshtein_ratio,
        CAST(({address_jaccard}) AS FLOAT) AS address_token_jaccard,
        CAST(LEAST(LENGTH(s1.address_norm), LENGTH(target.address_norm))::DOUBLE /
             GREATEST(LENGTH(s1.address_norm), LENGTH(target.address_norm), 1) AS FLOAT)
             AS address_length_ratio,

        CAST(s1.address_numbers <> '' AND s1.address_numbers = target.address_numbers AS FLOAT)
             AS address_numbers_exact,
        CAST(({number_jaccard}) AS FLOAT) AS address_number_jaccard,
        CAST(s1.address_numbers <> '' AND target.address_numbers <> ''
             AND ({number_jaccard}) = 0 AS FLOAT) AS address_number_conflict,

        CAST(s1.house_number <> '' AND s1.house_number = target.house_number AS FLOAT)
             AS house_number_exact,
        CAST(s1.house_number <> '' AND target.house_number <> ''
             AND s1.house_number <> target.house_number AS FLOAT) AS house_number_conflict,
        CAST(s1.postal_code <> '' AND s1.postal_code = target.postal_code AS FLOAT)
             AS postal_code_exact,
        CAST(s1.postal_code <> '' AND target.postal_code <> ''
             AND s1.postal_code <> target.postal_code AS FLOAT) AS postal_code_conflict,
        CAST(s1.name_norm = '' OR target.name_norm = '' AS FLOAT) AS name_missing,
        CAST(s1.address_norm = '' OR target.address_norm = '' AS FLOAT) AS address_missing,
        {label_sql}
    """


def build_feature_shards(
    label,
    candidate_shard_directory,
    s1_file,
    s2_file,
    s3_file,
    truth_file,
    local_feature_directory,
    persistent_feature_directory,
):
    local_feature_directory.mkdir(parents=True, exist_ok=True)
    persistent_feature_directory.mkdir(parents=True, exist_ok=True)
    include_label = truth_file is not None
    truth_join = ""
    if include_label:
        truth_join = f"""
            LEFT JOIN read_parquet('{Path(truth_file).as_posix()}') truth
              ON candidates.source1_entity_id = truth.source1_entity_id
             AND candidates.candidate_entity_id = truth.candidate_entity_id
        """
    select_sql = similarity_feature_sql(include_label)
    connection = create_connection(LOCAL_TEMP / f"{label}_features", memory_limit="5GB", threads=2)
    total_rows = 0
    overall_started = time.perf_counter()

    progress = tqdm(range(FEATURE_PARTITIONS), desc=f"{label} feature shards")
    for partition in progress:
        local_output = local_feature_directory / f"{label}_features_{partition:03d}.parquet"
        drive_output = persistent_feature_directory / local_output.name
        if restore_file(drive_output, local_output):
            rows = pq.ParquetFile(local_output).metadata.num_rows
            total_rows += rows
            progress.set_postfix(restored=partition, rows=f"{total_rows:,}")
            continue

        candidate_file = (
            candidate_shard_directory / f"{label}_candidates_{partition:03d}.parquet"
        )
        if not candidate_file.exists():
            raise FileNotFoundError(candidate_file)
        started = time.perf_counter()
        connection.execute(f"""
            COPY (
                WITH targets AS (
                    SELECT * FROM read_parquet('{Path(s2_file).as_posix()}')
                    UNION ALL
                    SELECT * FROM read_parquet('{Path(s3_file).as_posix()}')
                )
                SELECT {select_sql}
                FROM read_parquet('{candidate_file.as_posix()}') candidates
                INNER JOIN read_parquet('{Path(s1_file).as_posix()}') s1
                  ON candidates.source1_entity_id = s1.entity_id
                INNER JOIN targets target
                  ON candidates.candidate_entity_id = target.entity_id
                {truth_join}
            ) TO '{local_output.as_posix()}'
            (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
        """)
        rows = pq.ParquetFile(local_output).metadata.num_rows
        total_rows += rows
        atomic_copy(local_output, drive_output)
        progress.set_postfix(
            partition=partition,
            rows=f"{total_rows:,}",
            minutes=f"{(time.perf_counter()-started)/60:.1f}",
        )

    connection.close()
    print(
        f"{label} features complete | rows={total_rows:,} | "
        f"minutes={(time.perf_counter()-overall_started)/60:.1f}"
    )
    return local_feature_directory / f"{label}_features_*.parquet"


TRAIN_FEATURES_LOCAL = LOCAL_TRAIN / "feature_shards"
TRAIN_FEATURES_DRIVE = PERSIST_TRAIN / "feature_shards"
TRAIN_FEATURE_PATTERN = build_feature_shards(
    label="train",
    candidate_shard_directory=TRAIN_CANDIDATE_SHARDS_LOCAL,
    s1_file=TRAIN_NORMALIZED_FILES["s1"],
    s2_file=TRAIN_NORMALIZED_FILES["s2"],
    s3_file=TRAIN_NORMALIZED_FILES["s3"],
    truth_file=TRUTH_PAIRS_FILE,
    local_feature_directory=TRAIN_FEATURES_LOCAL,
    persistent_feature_directory=TRAIN_FEATURES_DRIVE,
)


train feature shards:   0%|          | 0/32 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/feature_shards/train_features_000.parquet | size GB: 0.141 | minutes: 0.01


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/feature_shards/train_features_001.parquet | size GB: 0.141 | minutes: 0.02


...

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/feature_shards/train_features_029.parquet | size GB: 0.142 | minutes: 0.01


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/feature_shards/train_features_030.parquet | size GB: 0.141 | minutes: 0.01


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/feature_shards/train_features_031.parquet | size GB: 0.141 | minutes: 0.01
train features complete | rows=172,564,964 | minutes=118.3


## Block 14 — Define and audit classifier features


In [14]:
FEATURE_COLUMNS = [
    "sn_min_rank_distance", "sn_pass_count",
    *PASS_FEATURE_COLUMNS,
    "candidate_source_is_s3",
    "name_exact", "name_core_exact", "name_acronym_exact",
    "name_jaro_winkler", "name_core_jaro_winkler",
    "name_levenshtein_ratio", "name_token_jaccard", "name_length_ratio",
    "address_exact", "address_jaro_winkler", "address_levenshtein_ratio",
    "address_token_jaccard", "address_length_ratio",
    "address_numbers_exact", "address_number_jaccard", "address_number_conflict",
    "house_number_exact", "house_number_conflict",
    "postal_code_exact", "postal_code_conflict",
    "name_missing", "address_missing",
]

connection = create_connection()
feature_summary = connection.execute(f"""
    SELECT split, COUNT(*) AS rows, SUM(label) AS positives,
           AVG(label) AS positive_rate
    FROM read_parquet('{TRAIN_FEATURE_PATTERN.as_posix()}')
    GROUP BY split ORDER BY split
""").df()

audit_expressions = ",\n".join(
    f"MIN({column}) AS {column}__min, MAX({column}) AS {column}__max, "
    f"AVG({column}) AS {column}__mean"
    for column in FEATURE_COLUMNS
)
feature_audit = connection.execute(f"""
    SELECT {audit_expressions}
    FROM read_parquet('{TRAIN_FEATURE_PATTERN.as_posix()}')
    WHERE split = 'model_train'
""").df()
connection.close()

display(feature_summary)
constant_features = []
for column in FEATURE_COLUMNS:
    minimum = feature_audit.iloc[0][f"{column}__min"]
    maximum = feature_audit.iloc[0][f"{column}__max"]
    if minimum == maximum:
        constant_features.append(column)
print("Feature count:", len(FEATURE_COLUMNS))
print("Constant features:", constant_features)
print("Numeric/postal audit:")
for column in [
    "address_numbers_exact", "address_number_jaccard", "address_number_conflict",
    "house_number_exact", "house_number_conflict", "postal_code_exact", "postal_code_conflict",
]:
    print(
        column,
        "min=", feature_audit.iloc[0][f"{column}__min"],
        "max=", feature_audit.iloc[0][f"{column}__max"],
        "mean=", feature_audit.iloc[0][f"{column}__mean"],
    )


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,rows,positives,positive_rate
0,model_train,115112244,2752908.0,0.023915
1,validation,57452720,1375169.0,0.023936


Feature count: 33
Constant features: ['name_missing']
Numeric/postal audit:
address_numbers_exact min= 0.0 max= 1.0 mean= 0.24493126899689316
address_number_jaccard min= 0.0 max= 1.0 mean= 0.30965982902680644
address_number_conflict min= 0.0 max= 1.0 mean= 0.5066822431156845
house_number_exact min= 0.0 max= 1.0 mean= 0.3393305494070639
house_number_conflict min= 0.0 max= 1.0 mean= 0.5823166213317846
postal_code_exact min= 0.0 max= 1.0 mean= 0.012577428340290195
postal_code_conflict min= 0.0 max= 1.0 mean= 0.016650435552277132


## Block 15 — Load positives and bounded random negatives


In [18]:
connection = create_connection(memory_limit="5GB", threads=2)
feature_path = TRAIN_FEATURE_PATTERN.as_posix()
positive_count = connection.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{feature_path}')
    WHERE split='model_train' AND label=1
""").fetchone()[0]
negative_count = connection.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{feature_path}')
    WHERE split='model_train' AND label=0
""").fetchone()[0]
negative_cutoff = min(
    1_000_000,
    max(1, int(MAX_TRAIN_NEGATIVES / max(1, negative_count) * 1_000_000)),
)
columns_sql = ", ".join(FEATURE_COLUMNS)
print("Available positives:", f"{positive_count:,}")
print("Available negatives:", f"{negative_count:,}")
print("Random-negative hash cutoff:", negative_cutoff)

base_training = connection.execute(f"""
    SELECT {columns_sql}, label
    FROM read_parquet('{feature_path}')
    WHERE split='model_train'
      AND (
          label=1 OR
          MOD(hash(source1_entity_id || '|' || candidate_entity_id), 1000000)
          < {negative_cutoff}
      )
""").df()
connection.close()

X_base = base_training[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
y_base = base_training["label"].astype(np.int8)
print("Base rows:", f"{len(X_base):,}")
print("Base positives:", f"{int(y_base.sum()):,}")
print("Base negatives:", f"{int((y_base == 0).sum()):,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Available positives: 2,752,908
Available negatives: 112,359,336
Random-negative hash cutoff: 26700


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rows: 5,750,317
Base positives: 2,752,908
Base negatives: 2,997,409


## Block 16 — Baseline model, hard-negative mining and final LightGBM


In [19]:
def format_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def progress_callback(total_trees, every=25, label="model"):
    started = time.perf_counter()
    def callback(environment):
        tree = environment.iteration + 1
        if tree == 1 or tree % every == 0 or tree == total_trees:
            elapsed = time.perf_counter() - started
            eta = elapsed / tree * (total_trees - tree)
            print(
                f"[{label}] tree {tree}/{total_trees} | "
                f"{100*tree/total_trees:6.2f}% | elapsed {format_duration(elapsed)} | "
                f"ETA {format_duration(eta)}",
                flush=True,
            )
    callback.order = 20
    callback.before_iteration = False
    return callback


def make_classifier(number_of_trees):
    return lgb.LGBMClassifier(
        objective="binary",
        n_estimators=number_of_trees,
        learning_rate=0.035,
        num_leaves=63,
        min_child_samples=250,
        max_bin=255,
        subsample=0.90,
        subsample_freq=1,
        colsample_bytree=0.90,
        reg_alpha=0.3,
        reg_lambda=3.0,
        random_state=SEED,
        n_jobs=-1,
        force_col_wise=True,
        verbosity=-1,
    )


base_weights = np.where(y_base.to_numpy() == 0, NEGATIVE_PAIR_WEIGHT, 1.0).astype(np.float32)
baseline_model = make_classifier(BASELINE_TREES)
print("Training baseline used only to locate hard false positives...")
baseline_model.fit(
    X_base,
    y_base,
    sample_weight=base_weights,
    callbacks=[progress_callback(BASELINE_TREES, label="baseline")],
)

HARD_NEGATIVE_FILE = LOCAL_TRAIN / "hard_negatives.parquet"
if HARD_NEGATIVE_FILE.exists():
    HARD_NEGATIVE_FILE.unlink()

connection = create_connection(memory_limit="4GB", threads=2)
reader = connection.execute(f"""
    SELECT {', '.join(FEATURE_COLUMNS)}
    FROM read_parquet('{TRAIN_FEATURE_PATTERN.as_posix()}')
    WHERE split='model_train' AND label=0
""").fetch_record_batch(rows_per_batch=250000)

hard_writer = None
scanned_negatives = 0
mined_negatives = 0
mining_started = time.perf_counter()
for batch in reader:
    frame = batch.to_pandas()
    X_batch = frame[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
    probabilities = baseline_model.predict_proba(X_batch)[:, 1]
    keep_indices = np.flatnonzero(probabilities >= HARD_NEGATIVE_MIN_PROBABILITY)
    remaining = MAX_HARD_NEGATIVES - mined_negatives
    if len(keep_indices) > remaining:
        local_order = np.argsort(probabilities[keep_indices])[-remaining:]
        keep_indices = keep_indices[local_order]
    if len(keep_indices):
        selected = frame.iloc[keep_indices][FEATURE_COLUMNS].reset_index(drop=True)
        table = pa.Table.from_pandas(selected, preserve_index=False)
        if hard_writer is None:
            hard_writer = pq.ParquetWriter(HARD_NEGATIVE_FILE, table.schema, compression="zstd")
        hard_writer.write_table(table, row_group_size=100000)
        mined_negatives += len(selected)
        del selected, table
    scanned_negatives += len(frame)
    print(
        f"Hard-negative mining: scanned={scanned_negatives:,} "
        f"mined={mined_negatives:,}/{MAX_HARD_NEGATIVES:,} "
        f"elapsed={(time.perf_counter()-mining_started)/60:.1f} min",
        flush=True,
    )
    del frame, X_batch, probabilities, keep_indices
    gc.collect()
    if mined_negatives >= MAX_HARD_NEGATIVES:
        break
if hard_writer is not None:
    hard_writer.close()
connection.close()

if mined_negatives:
    hard_frame = pd.read_parquet(HARD_NEGATIVE_FILE)
    X_hard = hard_frame[FEATURE_COLUMNS].astype(np.float32)
    X_final = pd.concat([X_base, X_hard], ignore_index=True)
    y_final = np.concatenate([y_base.to_numpy(), np.zeros(len(X_hard), dtype=np.int8)])
    final_weights = np.concatenate([
        base_weights,
        np.full(len(X_hard), HARD_NEGATIVE_WEIGHT, dtype=np.float32),
    ])
else:
    X_final = X_base
    y_final = y_base.to_numpy()
    final_weights = base_weights

print("Final training rows:", f"{len(X_final):,}")
print("Mined hard negatives:", f"{mined_negatives:,}")

final_model = make_classifier(FINAL_TREES)
training_started = time.perf_counter()
final_model.fit(
    X_final,
    y_final,
    sample_weight=final_weights,
    callbacks=[progress_callback(FINAL_TREES, label="final")],
)

MODEL_FILE = LOCAL_TRAIN / "enhanced_lightgbm.txt"
MODEL_DRIVE_FILE = PERSIST_MODELS / "enhanced_lightgbm.txt"
final_model.booster_.save_model(MODEL_FILE.as_posix())
atomic_copy(MODEL_FILE, MODEL_DRIVE_FILE)
print("Final training time:", format_duration(time.perf_counter()-training_started))
print("Model:", MODEL_DRIVE_FILE)

del base_training, X_base, y_base, base_weights, X_final, y_final, final_weights
if "hard_frame" in globals():
    del hard_frame, X_hard
gc.collect()


Training baseline used only to locate hard false positives...
[baseline] tree 1/150 |   0.67% | elapsed 00:00:07 | ETA 00:19:02
[baseline] tree 25/150 |  16.67% | elapsed 00:00:31 | ETA 00:02:35
[baseline] tree 50/150 |  33.33% | elapsed 00:00:56 | ETA 00:01:52
[baseline] tree 75/150 |  50.00% | elapsed 00:01:20 | ETA 00:01:20
[baseline] tree 100/150 |  66.67% | elapsed 00:01:46 | ETA 00:00:53
[baseline] tree 125/150 |  83.33% | elapsed 00:02:14 | ETA 00:00:26
[baseline] tree 150/150 | 100.00% | elapsed 00:02:42 | ETA 00:00:00


/tmp/ipykernel_1082/1867476369.py:65: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  """).fetch_record_batch(rows_per_batch=250000)


Hard-negative mining: scanned=250,000 mined=3,874/2,000,000 elapsed=0.0 min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Hard-negative mining: scanned=500,000 mined=7,804/2,000,000 elapsed=0.1 min
Hard-negative mining: scanned=750,000 mined=11,744/2,000,000 elapsed=0.1 min
Hard-negative mining: scanned=1,000,000 mined=15,723/2,000,000 elapsed=0.1 min
Hard-negative mining: scanned=1,250,000 mined=19,708/2,000,000 elapsed=0.2 min
Hard-negative mining: scanned=1,500,000 mined=23,670/2,000,000 elapsed=0.2 min
Hard-negative mining: scanned=1,750,000 mined=27,611/2,000,000 elapsed=0.2 min
Hard-negative mining: scanned=2,000,000 mined=31,821/2,000,000 elapsed=0.2 min
Hard-negative mining: scanned=2,250,000 mined=36,008/2,000,000 elapsed=0.3 min
Hard-negative mining: scanned=2,500,000 mined=40,329/2,000,000 elapsed=0.3 min
Hard-negative mining: scanned=2,750,000 mined=44,506/2,000,000 elapsed=0.4 min
Hard-negative mining: scanned=3,000,000 mined=48,676/2,000,000 elapsed=0.4 min
Hard-negative mining: scanned=3,250,000 mined=52,907/2,000,000 elapsed=0.4 min
Hard-negative mining: scanned=3,500,000 mined=57,088/2,00

53

## Block 17 — Resumable validation predictions


In [20]:
VALIDATION_PREDICTIONS_LOCAL = LOCAL_TRAIN / "validation_predictions"
VALIDATION_PREDICTIONS_DRIVE = PERSIST_TRAIN / "validation_predictions"
VALIDATION_PREDICTIONS_LOCAL.mkdir(parents=True, exist_ok=True)
VALIDATION_PREDICTIONS_DRIVE.mkdir(parents=True, exist_ok=True)

for partition in tqdm(range(FEATURE_PARTITIONS), desc="Validation prediction shards"):
    local_output = VALIDATION_PREDICTIONS_LOCAL / f"validation_predictions_{partition:03d}.parquet"
    drive_output = VALIDATION_PREDICTIONS_DRIVE / local_output.name
    if restore_file(drive_output, local_output):
        continue
    feature_file = TRAIN_FEATURES_LOCAL / f"train_features_{partition:03d}.parquet"
    connection = create_connection(memory_limit="3GB", threads=2)
    reader = connection.execute(f"""
        SELECT source1_entity_id, candidate_entity_id,
               {', '.join(FEATURE_COLUMNS)}, label
        FROM read_parquet('{feature_file.as_posix()}')
        WHERE split='validation'
    """).fetch_record_batch(rows_per_batch=200000)
    writer = None
    rows = 0
    for batch in reader:
        frame = batch.to_pandas()
        X_batch = frame[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
        probability = final_model.predict_proba(X_batch)[:, 1].astype(np.float32)
        table = pa.Table.from_pydict({
            "source1_entity_id": frame["source1_entity_id"],
            "candidate_entity_id": frame["candidate_entity_id"],
            "candidate_source_is_s3": frame["candidate_source_is_s3"].astype(np.float32),
            "label": frame["label"].astype(np.int8),
            "match_probability": probability,
        })
        if writer is None:
            writer = pq.ParquetWriter(local_output, table.schema, compression="zstd")
        writer.write_table(table, row_group_size=100000)
        rows += len(frame)
        del frame, X_batch, probability, table
        gc.collect()
    if writer is None:
        empty = pa.table({
            "source1_entity_id": pa.array([], type=pa.string()),
            "candidate_entity_id": pa.array([], type=pa.string()),
            "candidate_source_is_s3": pa.array([], type=pa.float32()),
            "label": pa.array([], type=pa.int8()),
            "match_probability": pa.array([], type=pa.float32()),
        })
        pq.write_table(empty, local_output, compression="zstd")
    else:
        writer.close()
    connection.close()
    atomic_copy(local_output, drive_output)
    print(f"Validation shard {partition:03d}: {rows:,} rows")

VALIDATION_PREDICTION_PATTERN = VALIDATION_PREDICTIONS_LOCAL / "validation_predictions_*.parquet"


Validation prediction shards:   0%|          | 0/32 [00:00<?, ?it/s]

/tmp/ipykernel_1082/3504106098.py:18: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  """).fetch_record_batch(rows_per_batch=200000)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/validation_predictions/validation_predictions_002.parquet | size GB: 0.024 | minutes: 0.0
Validation shard 002: 1,811,420 rows


/tmp/ipykernel_1082/3504106098.py:18: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  """).fetch_record_batch(rows_per_batch=200000)


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/validation_predictions/validation_predictions_030.parquet | size GB: 0.023 | minutes: 0.0
Validation shard 030: 1,778,160 rows


/tmp/ipykernel_1082/3504106098.py:18: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  """).fetch_record_batch(rows_per_batch=200000)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/training/validation_predictions/validation_predictions_031.parquet | size GB: 0.024 | minutes: 0.0
Validation shard 031: 1,814,922 rows


## Block 18 — Tune global then separate S2/S3 thresholds


In [21]:
def evaluate_thresholds(connection, threshold_s2, threshold_s3):
    row = connection.execute(f"""
        WITH validation_entities AS (
            SELECT entity_id AS source1_entity_id
            FROM read_parquet('{TRAIN_S1_FLAGGED_FILE.as_posix()}')
            WHERE split_code=1
        ),
        truth_counts AS (
            SELECT v.source1_entity_id,
                   COUNT(DISTINCT t.candidate_entity_id) AS truth_count
            FROM validation_entities v
            LEFT JOIN read_parquet('{TRUTH_PAIRS_FILE.as_posix()}') t USING(source1_entity_id)
            GROUP BY v.source1_entity_id
        ),
        predicted_counts AS (
            SELECT source1_entity_id, COUNT(*) AS predicted_count, SUM(label) AS tp
            FROM read_parquet('{VALIDATION_PREDICTION_PATTERN.as_posix()}')
            WHERE
                (candidate_source_is_s3 < 0.5 AND match_probability >= {float(threshold_s2)})
                OR
                (candidate_source_is_s3 >= 0.5 AND match_probability >= {float(threshold_s3)})
            GROUP BY source1_entity_id
        ),
        per_entity AS (
            SELECT tc.source1_entity_id, tc.truth_count,
                   COALESCE(pc.predicted_count,0) AS predicted_count,
                   COALESCE(pc.tp,0) AS tp,
                   CASE
                     WHEN tc.truth_count=0 THEN CASE WHEN COALESCE(pc.predicted_count,0)=0 THEN 1.0 ELSE 0.0 END
                     WHEN COALESCE(pc.tp,0)=0 THEN 0.0
                     ELSE 1.25*pc.tp /
                          (1.25*pc.tp + 0.25*(tc.truth_count-pc.tp) + (pc.predicted_count-pc.tp))
                   END AS entity_f0_5
            FROM truth_counts tc LEFT JOIN predicted_counts pc USING(source1_entity_id)
        )
        SELECT AVG(entity_f0_5),
               SUM(tp)::DOUBLE/NULLIF(SUM(predicted_count),0),
               SUM(tp)::DOUBLE/NULLIF(SUM(truth_count),0),
               SUM(predicted_count), SUM(tp)
        FROM per_entity
    """).fetchone()
    return {
        "threshold_s2": float(threshold_s2),
        "threshold_s3": float(threshold_s3),
        "macro_f0_5": float(row[0]),
        "pair_precision": float(row[1] or 0),
        "pair_recall": float(row[2] or 0),
        "predicted_pairs": int(row[3] or 0),
        "true_positive_pairs": int(row[4] or 0),
    }


connection = create_connection(memory_limit="5GB", threads=2)
global_results = []
for threshold in tqdm(np.arange(0.20, 0.96, 0.05), desc="Global coarse thresholds"):
    global_results.append(evaluate_thresholds(connection, threshold, threshold))
coarse_best = max(global_results, key=lambda row: row["macro_f0_5"])["threshold_s2"]
for threshold in tqdm(
    np.arange(max(0.01, coarse_best-0.06), min(0.99, coarse_best+0.061), 0.01),
    desc="Global fine thresholds",
):
    global_results.append(evaluate_thresholds(connection, threshold, threshold))
global_best = max(global_results, key=lambda row: row["macro_f0_5"])

offsets = [-0.08, -0.04, 0.0, 0.04, 0.08]
joint_results = []
for offset_s2 in tqdm(offsets, desc="Joint S2/S3 threshold rows"):
    for offset_s3 in offsets:
        t2 = min(0.999, max(0.001, global_best["threshold_s2"] + offset_s2))
        t3 = min(0.999, max(0.001, global_best["threshold_s3"] + offset_s3))
        joint_results.append(evaluate_thresholds(connection, t2, t3))
connection.close()

threshold_results = pd.DataFrame(global_results + joint_results).drop_duplicates(
    ["threshold_s2", "threshold_s3"]
).sort_values("macro_f0_5", ascending=False).reset_index(drop=True)
best_result = threshold_results.iloc[0].to_dict()
display(threshold_results.head(20))
print("Selected S2 threshold:", best_result["threshold_s2"])
print("Selected S3 threshold:", best_result["threshold_s3"])
print("Validation macro F0.5:", best_result["macro_f0_5"])
print("Validation precision:", best_result["pair_precision"])
print("Validation recall:", best_result["pair_recall"])


Global coarse thresholds:   0%|          | 0/16 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,threshold_s2,threshold_s3,macro_f0_5,pair_precision,pair_recall,predicted_pairs,true_positive_pairs
0,0.37,0.37,0.894731,0.976140,0.806894,1262308,1232189
1,0.33,0.37,0.894709,0.974353,0.809716,1269045,1236498
2,0.38,0.38,0.894703,0.976851,0.805678,1259488,1230332
3,0.36,0.36,0.894688,0.975286,0.808332,1265665,1234385
4,0.35,0.35,0.894667,0.974332,0.809862,1269302,1236721
5,0.35,0.35,0.894667,0.974332,0.809862,1269302,1236721
6,0.39,0.39,0.894665,0.977589,0.804460,1256635,1228472
7,0.41,0.37,0.894581,0.977504,0.804608,1256975,1228698
8,0.37,0.41,0.894579,0.977482,0.804349,1256599,1228303
9,0.34,0.34,0.894565,0.973320,0.811370,1272988,1239025


Selected S2 threshold: 0.37000000000000005
Selected S3 threshold: 0.37000000000000005
Validation macro F0.5: 0.8947305071737199
Validation precision: 0.9761397376868403
Validation recall: 0.8068938239525577


## Block 19 — Diagnostics and save model metadata


In [22]:
feature_importance = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance_gain": final_model.booster_.feature_importance(importance_type="gain"),
    "importance_split": final_model.booster_.feature_importance(importance_type="split"),
}).sort_values("importance_gain", ascending=False)
display(feature_importance)

METRICS_FILE = LOCAL_TRAIN / "enhanced_validation_metrics.json"
METRICS_DRIVE_FILE = PERSIST_MODELS / METRICS_FILE.name
THRESHOLDS_FILE = LOCAL_TRAIN / "enhanced_threshold_results.csv"
IMPORTANCE_FILE = LOCAL_TRAIN / "enhanced_feature_importance.csv"

final_metrics = {
    "method": "enhanced_multi_pass_sorted_neighborhood_hard_negative_lightgbm",
    "sort_passes": SORT_PASSES,
    "model_train_query_sample_modulus": MODEL_TRAIN_QUERY_SAMPLE_MODULUS,
    "candidate_buckets": CANDIDATE_BUCKETS,
    "baseline_trees": BASELINE_TREES,
    "final_trees": FINAL_TREES,
    "hard_negatives": int(mined_negatives),
    "best_validation_result": {
        key: value.item() if isinstance(value, np.generic) else value
        for key, value in best_result.items()
    },
    "candidate_metrics": {
        column: (
            candidate_metrics.iloc[0][column].item()
            if isinstance(candidate_metrics.iloc[0][column], np.generic)
            else candidate_metrics.iloc[0][column]
        ) for column in candidate_metrics.columns
    },
    "feature_columns": FEATURE_COLUMNS,
}
with open(METRICS_FILE, "w", encoding="utf-8") as stream:
    json.dump(final_metrics, stream, indent=2)
threshold_results.to_csv(THRESHOLDS_FILE, index=False)
feature_importance.to_csv(IMPORTANCE_FILE, index=False)
for artifact in [MODEL_FILE, METRICS_FILE, THRESHOLDS_FILE, IMPORTANCE_FILE]:
    atomic_copy(artifact, PERSIST_MODELS / artifact.name)
print("Training and validation complete.")
print("Validation macro F0.5:", best_result["macro_f0_5"])
print("Persistent model directory:", PERSIST_MODELS)


,feature,importance_gain,importance_split
22,address_token_jaccard,3.398882e+07,2791
1,sn_pass_count,2.710122e+07,380
14,name_jaro_winkler,1.591320e+07,2700
25,address_number_jaccard,1.521674e+07,1239
0,sn_min_rank_distance,1.413770e+07,702
8,sn_from_postal_house,8.544361e+06,334
17,name_token_jaccard,5.842107e+06,3059
16,name_levenshtein_ratio,5.614685e+06,2647
18,name_length_ratio,4.521337e+06,2541
15,name_core_jaro_winkler,4.136692e+06,2359


Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/models/enhanced_lightgbm.txt | size GB: 0.003 | minutes: 0.0
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/models/enhanced_validation_metrics.json | size GB: 0.0 | minutes: 0.0
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/models/enhanced_threshold_results.csv | size GB: 0.0 | minutes: 0.0
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/models/enhanced_feature_importance.csv | size GB: 0.0 | minutes: 0.0
Training and validation complete.
Validation macro F0.5: 0.8947305071737199
Persistent model directory: /content/drive/MyDrive/AMLC Second Try Shared/models


## Block 20 — Normalize test data and create test S1 query flags


In [23]:
test_inputs = {
    "s1": TEST_DIR / "test_source1.tsv",
    "s2": TEST_DIR / "test_source2.tsv",
    "s3": TEST_DIR / "test_source3.tsv",
}
for source in ["s1", "s2", "s3"]:
    normalize_tsv_to_parquet(
        test_inputs[source],
        TEST_NORMALIZED_FILES[source],
        TEST_NORMALIZED_DRIVE / TEST_NORMALIZED_FILES[source].name,
        f"test {source.upper()}",
    )

TEST_S1_FLAGGED_FILE = LOCAL_TEST / "test_s1_flagged.parquet"
TEST_S1_FLAGGED_DRIVE = PERSIST_TEST / "test_s1_flagged.parquet"
if not restore_file(TEST_S1_FLAGGED_DRIVE, TEST_S1_FLAGGED_FILE):
    connection = create_connection()
    connection.execute(f"""
        COPY (
            SELECT *, CAST(-1 AS TINYINT) AS split_code,
                   CAST(TRUE AS BOOLEAN) AS emit_query
            FROM read_parquet('{TEST_NORMALIZED_FILES["s1"].as_posix()}')
        ) TO '{TEST_S1_FLAGGED_FILE.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 150000)
    """)
    connection.close()
    atomic_copy(TEST_S1_FLAGGED_FILE, TEST_S1_FLAGGED_DRIVE)

connection = create_connection()
TEST_COUNTRIES = [row[0] for row in connection.execute(f"""
    SELECT DISTINCT country_norm
    FROM read_parquet('{TEST_S1_FLAGGED_FILE.as_posix()}')
    WHERE country_norm <> '' ORDER BY country_norm
""").fetchall()]
test_s1_count = connection.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{TEST_S1_FLAGGED_FILE.as_posix()}')
""").fetchone()[0]
connection.close()
print("Test countries:", TEST_COUNTRIES)
print("Test S1 records:", f"{test_s1_count:,}")


test S1: chunk=1 rows=150,000 elapsed=0.1 min
test S1: chunk=2 rows=300,000 elapsed=0.2 min
test S1: chunk=3 rows=450,000 elapsed=0.3 min
test S1: chunk=4 rows=600,000 elapsed=0.4 min
test S1: chunk=5 rows=750,000 elapsed=0.5 min
test S1: chunk=6 rows=900,000 elapsed=0.6 min
test S1: chunk=7 rows=1,050,000 elapsed=0.7 min
test S1: chunk=8 rows=1,200,000 elapsed=0.8 min
test S1: chunk=9 rows=1,350,000 elapsed=0.9 min
test S1: chunk=10 rows=1,500,000 elapsed=1.0 min
test S1: chunk=11 rows=1,650,000 elapsed=1.1 min
test S1: chunk=12 rows=1,732,544 elapsed=1.1 min
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/normalized/test_s1.parquet | size GB: 0.327 | minutes: 0.03
test S2: chunk=1 rows=150,000 elapsed=0.1 min
test S2: chunk=2 rows=300,000 elapsed=0.2 min
test S2: chunk=3 rows=450,000 elapsed=0.3 min
test S2: chunk=4 rows=600,000 elapsed=0.5 min
test S2: chunk=5 rows=750,000 elapsed=0.5 min
test S2: chunk=6 rows=900,000 elapsed=0.7 min
test S2: chunk=7 rows=1,050,0

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/test_s1_flagged.parquet | size GB: 0.276 | minutes: 0.03
Test countries: ['france', 'india', 'us']
Test S1 records: 1,732,544


## Block 21 — Generate resumable enhanced test candidates


In [24]:
TEST_CANDIDATE_PARTS_LOCAL = LOCAL_TEST / "candidate_distance_parts"
TEST_CANDIDATE_PARTS_DRIVE = PERSIST_TEST / "candidate_distance_parts"
TEST_CANDIDATE_PARTS_LOCAL.mkdir(parents=True, exist_ok=True)
TEST_CANDIDATE_PARTS_DRIVE.mkdir(parents=True, exist_ok=True)

total_jobs = len(SORT_PASSES) * len(TEST_COUNTRIES)
job = 0
started = time.perf_counter()
for pass_name in SORT_PASSES:
    for country in TEST_COUNTRIES:
        job += 1
        print(f"\nTEST JOB {job}/{total_jobs}")
        generate_gpu_pass_for_country(
            label="test",
            pass_name=pass_name,
            country=country,
            s1_path=TEST_S1_FLAGGED_FILE,
            s2_path=TEST_NORMALIZED_FILES["s2"],
            s3_path=TEST_NORMALIZED_FILES["s3"],
            local_parts_dir=TEST_CANDIDATE_PARTS_LOCAL,
            persistent_parts_dir=TEST_CANDIDATE_PARTS_DRIVE,
        )
print("Test candidate-generation minutes:", round((time.perf_counter()-started)/60, 2))



TEST JOB 1/24

TEST | pass=name | country=france | radius=20
GPU before load: {'used_GB': 0.13, 'free_GB': 14.43, 'total_GB': 14.56}
Loaded S1=259,452, S2=703,378, S3=731,615
Sorted 1,694,445 rows in 0.00 min
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/candidate_distance_parts/test_gpu_00_name_france_d01.parquet | size GB: 0.009 | minutes: 0.0
distance 01/20: pairs=475,474 cumulative=475,474 seconds=0.4
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/candidate_distance_parts/test_gpu_00_name_france_d02.parquet | size GB: 0.008 | minutes: 0.0
distance 02/20: pairs=454,323 cumulative=929,797 seconds=0.4
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/candidate_distance_parts/test_gpu_00_name_france_d03.parquet | size GB: 0.008 | minutes: 0.0
distance 03/20: pairs=434,377 cumulative=1,364,174 seconds=0.3
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/candidate_distance_parts/test_gpu_00_name_france_d04.p

## Block 22 — Union test candidates into bounded shards


In [25]:
TEST_CANDIDATE_SHARDS_LOCAL = LOCAL_TEST / "candidate_shards"
TEST_CANDIDATE_SHARDS_DRIVE = PERSIST_TEST / "candidate_shards"

TEST_CANDIDATE_PATTERN = union_candidate_parts(
    label="test",
    parts_directory=TEST_CANDIDATE_PARTS_LOCAL,
    part_pattern="test_gpu_*.parquet",
    local_shard_directory=TEST_CANDIDATE_SHARDS_LOCAL,
    persistent_shard_directory=TEST_CANDIDATE_SHARDS_DRIVE,
    temporary_directory=LOCAL_TEMP / "test_union",
)

connection = create_connection()
test_candidate_count = connection.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{TEST_CANDIDATE_PATTERN.as_posix()}')
""").fetchone()[0]
test_candidate_stats = connection.execute(f"""
    WITH counts AS (
        SELECT source1_entity_id, COUNT(*) AS candidate_count
        FROM read_parquet('{TEST_CANDIDATE_PATTERN.as_posix()}')
        GROUP BY source1_entity_id
    )
    SELECT COUNT(*) AS s1_with_candidates,
           AVG(candidate_count) AS average_candidates,
           MEDIAN(candidate_count) AS median_candidates,
           QUANTILE_CONT(candidate_count,0.95) AS p95_candidates,
           MAX(candidate_count) AS maximum_candidates
    FROM counts
""").df()
connection.close()
print("Test candidate pairs:", f"{test_candidate_count:,}")
display(test_candidate_stats)


Partitioning 357 test distance files into 32 buckets...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw partition minutes: 3.66


Union test buckets:   0%|          | 0/32 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/candidate_shards/test_candidates_031.parquet | size GB: 0.07 | minutes: 0.01


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Test candidate pairs: 229,861,664


,s1_with_candidates,average_candidates,median_candidates,p95_candidates,maximum_candidates
0,1732544,132.672916,133.0,158.0,202


pseudo extra block

In [10]:
# CPU-only recovery after a runtime restart.
# Run after Blocks 1–5.

# ---------------------------------------------------------
# Restore normalized test files
# ---------------------------------------------------------

for source in ["s1", "s2", "s3"]:
    persistent_file = (
        TEST_NORMALIZED_DRIVE
        / TEST_NORMALIZED_FILES[source].name
    )

    restored = restore_file(
        persistent_file,
        TEST_NORMALIZED_FILES[source],
    )

    if not restored:
        raise FileNotFoundError(
            f"Missing normalized test checkpoint: "
            f"{persistent_file}"
        )


# ---------------------------------------------------------
# Restore flagged test S1
# ---------------------------------------------------------

TEST_S1_FLAGGED_FILE = (
    LOCAL_TEST / "test_s1_flagged.parquet"
)

TEST_S1_FLAGGED_DRIVE = (
    PERSIST_TEST / "test_s1_flagged.parquet"
)

if not restore_file(
    TEST_S1_FLAGGED_DRIVE,
    TEST_S1_FLAGGED_FILE,
):
    raise FileNotFoundError(
        f"Missing test S1 checkpoint: "
        f"{TEST_S1_FLAGGED_DRIVE}"
    )


# ---------------------------------------------------------
# Restore all 32 test candidate shards
# ---------------------------------------------------------

TEST_CANDIDATE_SHARDS_LOCAL = (
    LOCAL_TEST / "candidate_shards"
)

TEST_CANDIDATE_SHARDS_DRIVE = (
    PERSIST_TEST / "candidate_shards"
)

TEST_CANDIDATE_SHARDS_LOCAL.mkdir(
    parents=True,
    exist_ok=True,
)

missing_candidate_shards = []

for partition in range(CANDIDATE_BUCKETS):
    filename = (
        f"test_candidates_{partition:03d}.parquet"
    )

    local_file = (
        TEST_CANDIDATE_SHARDS_LOCAL
        / filename
    )

    drive_file = (
        TEST_CANDIDATE_SHARDS_DRIVE
        / filename
    )

    if not restore_file(
        drive_file,
        local_file,
    ):
        missing_candidate_shards.append(
            filename
        )

if missing_candidate_shards:
    raise FileNotFoundError(
        "Missing candidate shards in Drive: "
        + ", ".join(missing_candidate_shards)
    )

TEST_CANDIDATE_PATTERN = (
    TEST_CANDIDATE_SHARDS_LOCAL
    / "test_candidates_*.parquet"
)


# ---------------------------------------------------------
# Feature definitions needed by Blocks 23–25
# ---------------------------------------------------------

PASS_FEATURE_COLUMNS = [
    f"sn_from_{name}"
    for name in SORT_PASSES
]

FEATURE_COLUMNS = [
    "sn_min_rank_distance",
    "sn_pass_count",
    *PASS_FEATURE_COLUMNS,
    "candidate_source_is_s3",

    "name_exact",
    "name_core_exact",
    "name_acronym_exact",
    "name_jaro_winkler",
    "name_core_jaro_winkler",
    "name_levenshtein_ratio",
    "name_token_jaccard",
    "name_length_ratio",

    "address_exact",
    "address_jaro_winkler",
    "address_levenshtein_ratio",
    "address_token_jaccard",
    "address_length_ratio",

    "address_numbers_exact",
    "address_number_jaccard",
    "address_number_conflict",

    "house_number_exact",
    "house_number_conflict",
    "postal_code_exact",
    "postal_code_conflict",

    "name_missing",
    "address_missing",
]


def token_jaccard_sql(
    left_expression,
    right_expression,
):
    return f"""
        CASE
            WHEN
                {left_expression} = ''
                OR {right_expression} = ''
            THEN 0.0

            ELSE
                list_count(
                    list_intersect(
                        list_distinct(
                            string_split(
                                {left_expression},
                                ' '
                            )
                        ),

                        list_distinct(
                            string_split(
                                {right_expression},
                                ' '
                            )
                        )
                    )
                )::DOUBLE

                /

                NULLIF(
                    list_count(
                        list_distinct(
                            list_concat(
                                string_split(
                                    {left_expression},
                                    ' '
                                ),

                                string_split(
                                    {right_expression},
                                    ' '
                                )
                            )
                        )
                    ),
                    0
                )
        END
    """


def similarity_feature_sql(
    include_label,
):
    name_jaccard = token_jaccard_sql(
        "s1.name_core",
        "target.name_core",
    )

    address_jaccard = token_jaccard_sql(
        "s1.address_norm",
        "target.address_norm",
    )

    number_jaccard = token_jaccard_sql(
        "s1.address_numbers",
        "target.address_numbers",
    )

    pass_columns = ",\n".join(
        (
            f"CAST(candidates.{column} "
            f"AS FLOAT) AS {column}"
        )
        for column in PASS_FEATURE_COLUMNS
    )

    if include_label:
        label_sql = """
            CAST(
                truth.candidate_entity_id
                IS NOT NULL
                AS INTEGER
            ) AS label
        """
    else:
        label_sql = """
            CAST(0 AS INTEGER) AS label
        """

    return f"""
        candidates.source1_entity_id,
        candidates.candidate_entity_id,
        candidates.split,

        CAST(
            candidates.sn_min_rank_distance
            AS FLOAT
        ) AS sn_min_rank_distance,

        CAST(
            candidates.sn_pass_count
            AS FLOAT
        ) AS sn_pass_count,

        {pass_columns},

        CAST(
            candidates.candidate_source = 'S3'
            AS FLOAT
        ) AS candidate_source_is_s3,

        CAST(
            s1.name_norm <> ''
            AND s1.name_norm = target.name_norm
            AS FLOAT
        ) AS name_exact,

        CAST(
            s1.name_core <> ''
            AND s1.name_core = target.name_core
            AS FLOAT
        ) AS name_core_exact,

        CAST(
            s1.name_acronym <> ''
            AND
            s1.name_acronym
                = target.name_acronym
            AS FLOAT
        ) AS name_acronym_exact,

        CAST(
            jaro_winkler_similarity(
                s1.name_norm,
                target.name_norm
            )
            AS FLOAT
        ) AS name_jaro_winkler,

        CAST(
            jaro_winkler_similarity(
                s1.name_core,
                target.name_core
            )
            AS FLOAT
        ) AS name_core_jaro_winkler,

        CAST(
            1.0
            -
            levenshtein(
                s1.name_norm,
                target.name_norm
            )::DOUBLE

            /

            GREATEST(
                LENGTH(s1.name_norm),
                LENGTH(target.name_norm),
                1
            )
            AS FLOAT
        ) AS name_levenshtein_ratio,

        CAST(
            ({name_jaccard})
            AS FLOAT
        ) AS name_token_jaccard,

        CAST(
            LEAST(
                LENGTH(s1.name_norm),
                LENGTH(target.name_norm)
            )::DOUBLE

            /

            GREATEST(
                LENGTH(s1.name_norm),
                LENGTH(target.name_norm),
                1
            )
            AS FLOAT
        ) AS name_length_ratio,

        CAST(
            s1.address_norm <> ''
            AND
            s1.address_norm
                = target.address_norm
            AS FLOAT
        ) AS address_exact,

        CAST(
            jaro_winkler_similarity(
                s1.address_norm,
                target.address_norm
            )
            AS FLOAT
        ) AS address_jaro_winkler,

        CAST(
            1.0
            -
            levenshtein(
                s1.address_norm,
                target.address_norm
            )::DOUBLE

            /

            GREATEST(
                LENGTH(s1.address_norm),
                LENGTH(target.address_norm),
                1
            )
            AS FLOAT
        ) AS address_levenshtein_ratio,

        CAST(
            ({address_jaccard})
            AS FLOAT
        ) AS address_token_jaccard,

        CAST(
            LEAST(
                LENGTH(s1.address_norm),
                LENGTH(target.address_norm)
            )::DOUBLE

            /

            GREATEST(
                LENGTH(s1.address_norm),
                LENGTH(target.address_norm),
                1
            )
            AS FLOAT
        ) AS address_length_ratio,

        CAST(
            s1.address_numbers <> ''
            AND
            s1.address_numbers
                = target.address_numbers
            AS FLOAT
        ) AS address_numbers_exact,

        CAST(
            ({number_jaccard})
            AS FLOAT
        ) AS address_number_jaccard,

        CAST(
            s1.address_numbers <> ''
            AND target.address_numbers <> ''
            AND ({number_jaccard}) = 0
            AS FLOAT
        ) AS address_number_conflict,

        CAST(
            s1.house_number <> ''
            AND
            s1.house_number
                = target.house_number
            AS FLOAT
        ) AS house_number_exact,

        CAST(
            s1.house_number <> ''
            AND target.house_number <> ''
            AND
            s1.house_number
                <> target.house_number
            AS FLOAT
        ) AS house_number_conflict,

        CAST(
            s1.postal_code <> ''
            AND
            s1.postal_code
                = target.postal_code
            AS FLOAT
        ) AS postal_code_exact,

        CAST(
            s1.postal_code <> ''
            AND target.postal_code <> ''
            AND
            s1.postal_code
                <> target.postal_code
            AS FLOAT
        ) AS postal_code_conflict,

        CAST(
            s1.name_norm = ''
            OR target.name_norm = ''
            AS FLOAT
        ) AS name_missing,

        CAST(
            s1.address_norm = ''
            OR target.address_norm = ''
            AS FLOAT
        ) AS address_missing,

        {label_sql}
    """


def build_feature_shards(
    label,
    candidate_shard_directory,
    s1_file,
    s2_file,
    s3_file,
    truth_file,
    local_feature_directory,
    persistent_feature_directory,
):
    local_feature_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    persistent_feature_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    include_label = (
        truth_file is not None
    )

    truth_join = ""

    if include_label:
        truth_join = f"""
            LEFT JOIN read_parquet(
                '{Path(truth_file).as_posix()}'
            ) truth

                ON
                candidates.source1_entity_id
                    = truth.source1_entity_id

                AND

                candidates.candidate_entity_id
                    = truth.candidate_entity_id
        """

    select_sql = similarity_feature_sql(
        include_label
    )

    connection = create_connection(
        LOCAL_TEMP / f"{label}_features",
        memory_limit="5GB",
        threads=2,
    )

    total_rows = 0
    overall_started = time.perf_counter()

    progress = tqdm(
        range(FEATURE_PARTITIONS),
        desc=f"{label} feature shards",
    )

    for partition in progress:
        local_output = (
            local_feature_directory
            / (
                f"{label}_features_"
                f"{partition:03d}.parquet"
            )
        )

        drive_output = (
            persistent_feature_directory
            / local_output.name
        )

        # Completed Drive shards are restored and skipped.
        if restore_file(
            drive_output,
            local_output,
        ):
            rows = (
                pq.ParquetFile(
                    local_output
                )
                .metadata
                .num_rows
            )

            total_rows += rows

            progress.set_postfix(
                restored=partition,
                rows=f"{total_rows:,}",
            )

            continue

        candidate_file = (
            candidate_shard_directory
            / (
                f"{label}_candidates_"
                f"{partition:03d}.parquet"
            )
        )

        if not candidate_file.exists():
            raise FileNotFoundError(
                candidate_file
            )

        started = time.perf_counter()

        connection.execute(f"""
            COPY (
                WITH targets AS (
                    SELECT *
                    FROM read_parquet(
                        '{Path(s2_file).as_posix()}'
                    )

                    UNION ALL

                    SELECT *
                    FROM read_parquet(
                        '{Path(s3_file).as_posix()}'
                    )
                )

                SELECT
                    {select_sql}

                FROM read_parquet(
                    '{candidate_file.as_posix()}'
                ) candidates

                INNER JOIN read_parquet(
                    '{Path(s1_file).as_posix()}'
                ) s1

                    ON
                    candidates.source1_entity_id
                        = s1.entity_id

                INNER JOIN targets target

                    ON
                    candidates.candidate_entity_id
                        = target.entity_id

                {truth_join}
            )

            TO '{local_output.as_posix()}'

            (
                FORMAT PARQUET,
                COMPRESSION ZSTD,
                ROW_GROUP_SIZE 100000
            )
        """)

        rows = (
            pq.ParquetFile(
                local_output
            )
            .metadata
            .num_rows
        )

        total_rows += rows

        atomic_copy(
            local_output,
            drive_output,
        )

        progress.set_postfix(
            partition=partition,
            rows=f"{total_rows:,}",
            minutes=(
                f"{(
                    time.perf_counter()
                    - started
                ) / 60:.1f}"
            ),
        )

    connection.close()

    print(
        f"{label} features complete | "
        f"rows={total_rows:,} | "
        f"minutes="
        f"{(
            time.perf_counter()
            - overall_started
        ) / 60:.1f}"
    )

    return (
        local_feature_directory
        / f"{label}_features_*.parquet"
    )


# ---------------------------------------------------------
# Values required by later test blocks
# ---------------------------------------------------------

TEST_FEATURES_LOCAL = (
    LOCAL_TEST / "feature_shards"
)

TEST_FEATURES_DRIVE = (
    PERSIST_TEST / "feature_shards"
)

connection = create_connection()

test_s1_count = connection.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{TEST_NORMALIZED_FILES["s1"].as_posix()}'
    )
""").fetchone()[0]

test_candidate_count = connection.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{TEST_CANDIDATE_PATTERN.as_posix()}'
    )
""").fetchone()[0]

connection.close()

print("CPU recovery completed.")
print("Test S1:", f"{test_s1_count:,}")
print(
    "Restored test candidates:",
    f"{test_candidate_count:,}",
)
print(
    "Existing Drive feature shards:",
    len(
        list(
            TEST_FEATURES_DRIVE.glob(
                "test_features_*.parquet"
            )
        )
    ),
    "/",
    FEATURE_PARTITIONS,
)

Restoring: test_s1.parquet
Restoring: test_s2.parquet
Restoring: test_s3.parquet
Restoring: test_s1_flagged.parquet
Restoring: test_candidates_000.parquet
Restoring: test_candidates_001.parquet
Restoring: test_candidates_002.parquet
Restoring: test_candidates_003.parquet
Restoring: test_candidates_004.parquet
Restoring: test_candidates_005.parquet
Restoring: test_candidates_006.parquet
Restoring: test_candidates_007.parquet
Restoring: test_candidates_008.parquet
Restoring: test_candidates_009.parquet
Restoring: test_candidates_010.parquet
Restoring: test_candidates_011.parquet
Restoring: test_candidates_012.parquet
Restoring: test_candidates_013.parquet
Restoring: test_candidates_014.parquet
Restoring: test_candidates_015.parquet
Restoring: test_candidates_016.parquet
Restoring: test_candidates_017.parquet
Restoring: test_candidates_018.parquet
Restoring: test_candidates_019.parquet
Restoring: test_candidates_020.parquet
Restoring: test_candidates_021.parquet
Restoring: test_candidates

## Block 23 — Build resumable test feature shards


In [11]:
TEST_FEATURES_LOCAL = LOCAL_TEST / "feature_shards"
TEST_FEATURES_DRIVE = PERSIST_TEST / "feature_shards"
TEST_FEATURE_PATTERN = build_feature_shards(
    label="test",
    candidate_shard_directory=TEST_CANDIDATE_SHARDS_LOCAL,
    s1_file=TEST_NORMALIZED_FILES["s1"],
    s2_file=TEST_NORMALIZED_FILES["s2"],
    s3_file=TEST_NORMALIZED_FILES["s3"],
    truth_file=None,
    local_feature_directory=TEST_FEATURES_LOCAL,
    persistent_feature_directory=TEST_FEATURES_DRIVE,
)


test feature shards:   0%|          | 0/32 [00:00<?, ?it/s]

Restoring: test_features_000.parquet
Restoring: test_features_001.parquet
Restoring: test_features_002.parquet
Restoring: test_features_003.parquet
Restoring: test_features_004.parquet
Restoring: test_features_005.parquet
Restoring: test_features_006.parquet
Restoring: test_features_007.parquet
Restoring: test_features_008.parquet
Restoring: test_features_009.parquet
Restoring: test_features_010.parquet
Restoring: test_features_011.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/feature_shards/test_features_012.parquet | size GB: 0.189 | minutes: 0.02


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/feature_shards/test_features_029.parquet | size GB: 0.189 | minutes: 0.02


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/feature_shards/test_features_030.parquet | size GB: 0.188 | minutes: 0.03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/testing/feature_shards/test_features_031.parquet | size GB: 0.189 | minutes: 0.04
test features complete | rows=229,861,664 | minutes=133.9


## Block 24 — Load saved model and thresholds


In [12]:
MODEL_DRIVE_FILE = PERSIST_MODELS / "enhanced_lightgbm.txt"
METRICS_DRIVE_FILE = PERSIST_MODELS / "enhanced_validation_metrics.json"
if not MODEL_DRIVE_FILE.exists() or not METRICS_DRIVE_FILE.exists():
    raise FileNotFoundError("Enhanced model/metrics are missing. Complete Blocks 15–19.")
with open(METRICS_DRIVE_FILE, "r", encoding="utf-8") as stream:
    saved_metrics = json.load(stream)
if list(saved_metrics["feature_columns"]) != FEATURE_COLUMNS:
    raise RuntimeError("Saved model feature columns do not match this notebook.")
TEST_THRESHOLD_S2 = float(saved_metrics["best_validation_result"]["threshold_s2"])
TEST_THRESHOLD_S3 = float(saved_metrics["best_validation_result"]["threshold_s3"])
test_model = lgb.Booster(model_file=MODEL_DRIVE_FILE.as_posix())
if test_model.num_feature() != len(FEATURE_COLUMNS):
    raise RuntimeError(
        f"Model expects {test_model.num_feature()} features; notebook has {len(FEATURE_COLUMNS)}."
    )
print("Model loaded:", MODEL_DRIVE_FILE)
print("S2 threshold:", TEST_THRESHOLD_S2)
print("S3 threshold:", TEST_THRESHOLD_S3)


Model loaded: /content/drive/MyDrive/AMLC Second Try Shared/models/enhanced_lightgbm.txt
S2 threshold: 0.37000000000000005
S3 threshold: 0.37000000000000005


## Block 25 — Score test feature shards with Drive checkpoints


In [13]:
TEST_SELECTED_LOCAL = LOCAL_TEST / "selected_shards"
TEST_SELECTED_DRIVE = PERSIST_TEST / "selected_shards"
TEST_SELECTED_LOCAL.mkdir(parents=True, exist_ok=True)
TEST_SELECTED_DRIVE.mkdir(parents=True, exist_ok=True)

total_scored = 0
total_selected = 0
overall_started = time.perf_counter()
for partition in tqdm(range(FEATURE_PARTITIONS), desc="Scoring test shards"):
    local_output = TEST_SELECTED_LOCAL / f"test_selected_{partition:03d}.parquet"
    drive_output = TEST_SELECTED_DRIVE / local_output.name
    if restore_file(drive_output, local_output):
        metadata = pq.ParquetFile(local_output).metadata
        total_selected += metadata.num_rows
        print(f"Selected shard {partition:03d} restored: {metadata.num_rows:,} rows")
        continue

    feature_file = TEST_FEATURES_LOCAL / f"test_features_{partition:03d}.parquet"
    connection = create_connection(memory_limit="3GB", threads=2)
    reader = connection.execute(f"""
        SELECT source1_entity_id, candidate_entity_id,
               {', '.join(FEATURE_COLUMNS)}
        FROM read_parquet('{feature_file.as_posix()}')
    """).fetch_record_batch(rows_per_batch=200000)
    writer = None
    shard_scored = 0
    shard_selected = 0
    shard_started = time.perf_counter()
    for batch in reader:
        frame = batch.to_pandas()
        X_batch = frame[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)
        probability = test_model.predict(X_batch).astype(np.float32)
        source_is_s3 = frame["candidate_source_is_s3"].to_numpy() >= 0.5
        keep = np.where(
            source_is_s3,
            probability >= TEST_THRESHOLD_S3,
            probability >= TEST_THRESHOLD_S2,
        )
        if keep.any():
            table = pa.Table.from_pydict({
                "source1_entity_id": frame.loc[keep, "source1_entity_id"],
                "candidate_entity_id": frame.loc[keep, "candidate_entity_id"],
                "match_probability": probability[keep],
            })
            if writer is None:
                writer = pq.ParquetWriter(local_output, table.schema, compression="zstd")
            writer.write_table(table, row_group_size=100000)
            shard_selected += int(keep.sum())
            del table
        shard_scored += len(frame)
        print(
            f"test shard {partition:03d}: scored={shard_scored:,} "
            f"selected={shard_selected:,} elapsed={(time.perf_counter()-shard_started)/60:.1f} min",
            flush=True,
        )
        del frame, X_batch, probability, source_is_s3, keep
        gc.collect()
    if writer is None:
        empty = pa.table({
            "source1_entity_id": pa.array([], type=pa.string()),
            "candidate_entity_id": pa.array([], type=pa.string()),
            "match_probability": pa.array([], type=pa.float32()),
        })
        pq.write_table(empty, local_output, compression="zstd")
    else:
        writer.close()
    connection.close()
    atomic_copy(local_output, drive_output)
    total_scored += shard_scored
    total_selected += shard_selected

TEST_SELECTED_PATTERN = TEST_SELECTED_LOCAL / "test_selected_*.parquet"
print("Scored rows in newly computed shards:", f"{total_scored:,}")
print("Total selected/restored rows:", f"{total_selected:,}")
print("Scoring elapsed minutes:", round((time.perf_counter()-overall_started)/60, 2))


Scoring test shards:   0%|          | 0/32 [00:00<?, ?it/s]

/tmp/ipykernel_927/2731936697.py:24: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  """).fetch_record_batch(rows_per_batch=200000)


test shard 000: scored=200,000 selected=4,724 elapsed=0.1 min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

test shard 000: scored=400,000 selected=9,455 elapsed=0.3 min
test shard 000: scored=600,000 selected=14,225 elapsed=0.4 min
test shard 000: scored=800,000 selected=18,972 elapsed=0.5 min
test shard 000: scored=1,000,000 selected=23,606 elapsed=0.7 min
test shard 000: scored=1,200,000 selected=28,354 elapsed=0.8 min
test shard 000: scored=1,400,000 selected=33,056 elapsed=0.9 min
test shard 000: scored=1,600,000 selected=37,832 elapsed=1.0 min
test shard 000: scored=1,800,000 selected=42,494 elapsed=1.2 min
test shard 000: scored=2,000,000 selected=47,219 elapsed=1.3 min
test shard 000: scored=2,200,000 selected=51,990 elapsed=1.4 min
test shard 000: scored=2,400,000 selected=56,747 elapsed=1.5 min
test shard 000: scored=2,600,000 selected=61,666 elapsed=1.7 min
test shard 000: scored=2,800,000 selected=66,431 elapsed=1.8 min
test shard 000: scored=3,000,000 selected=71,219 elapsed=1.9 min
test shard 000: scored=3,200,000 selected=75,985 elapsed=2.0 min
test shard 000: scored=3,400,000

...

test shard 030: scored=400,000 selected=9,402 elapsed=0.3 min
test shard 030: scored=600,000 selected=14,156 elapsed=0.4 min
test shard 030: scored=800,000 selected=18,880 elapsed=0.5 min
test shard 030: scored=1,000,000 selected=23,624 elapsed=0.7 min
test shard 030: scored=1,200,000 selected=28,411 elapsed=0.8 min
test shard 030: scored=1,400,000 selected=33,233 elapsed=0.9 min
test shard 030: scored=1,600,000 selected=37,973 elapsed=1.0 min
test shard 030: scored=1,800,000 selected=42,871 elapsed=1.2 min
test shard 030: scored=2,000,000 selected=47,599 elapsed=1.3 min
test shard 030: scored=2,200,000 selected=52,324 elapsed=1.4 min
test shard 030: scored=2,400,000 selected=57,089 elapsed=1.5 min
test shard 030: scored=2,600,000 selected=61,709 elapsed=1.7 min
test shard 030: scored=2,800,000 selected=66,509 elapsed=1.8 min
test shard 030: scored=3,000,000 selected=71,161 elapsed=1.9 min
test shard 030: scored=3,200,000 selected=75,871 elapsed=2.1 min
test shard 030: scored=3,400,000

/tmp/ipykernel_927/2731936697.py:24: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  """).fetch_record_batch(rows_per_batch=200000)


test shard 031: scored=200,000 selected=4,939 elapsed=0.1 min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

test shard 031: scored=400,000 selected=9,695 elapsed=0.3 min
test shard 031: scored=600,000 selected=14,328 elapsed=0.4 min
test shard 031: scored=800,000 selected=19,087 elapsed=0.5 min
test shard 031: scored=1,000,000 selected=23,881 elapsed=0.6 min
test shard 031: scored=1,200,000 selected=28,553 elapsed=0.8 min
test shard 031: scored=1,400,000 selected=33,331 elapsed=0.9 min
test shard 031: scored=1,600,000 selected=38,162 elapsed=1.0 min
test shard 031: scored=1,800,000 selected=42,901 elapsed=1.1 min
test shard 031: scored=2,000,000 selected=47,736 elapsed=1.3 min
test shard 031: scored=2,200,000 selected=52,433 elapsed=1.4 min
test shard 031: scored=2,400,000 selected=57,167 elapsed=1.5 min
test shard 031: scored=2,600,000 selected=61,931 elapsed=1.6 min
test shard 031: scored=2,800,000 selected=66,678 elapsed=1.8 min
test shard 031: scored=3,000,000 selected=71,541 elapsed=1.9 min
test shard 031: scored=3,200,000 selected=76,426 elapsed=2.0 min
test shard 031: scored=3,400,000

## Block 26 — Generate both final TSV files without global aggregation


In [14]:
OUTPUT_LOCAL = LOCAL_TEST / "output"
OUTPUT_LOCAL.mkdir(parents=True, exist_ok=True)
MATCHING_RESULTS_FILE = OUTPUT_LOCAL / "matching_results.tsv"
CANDIDATE_PAIRS_FILE = OUTPUT_LOCAL / "candidate_pairs.tsv"

S1_BUCKET_DIR = LOCAL_TEST / "s1_export_buckets"
if S1_BUCKET_DIR.exists():
    shutil.rmtree(S1_BUCKET_DIR)

connection = create_connection(LOCAL_TEMP / "final_export", memory_limit="3GB", threads=1)
connection.execute(f"""
    COPY (
        SELECT entity_id,
               MOD(hash(entity_id), {CANDIDATE_BUCKETS}) AS candidate_bucket
        FROM read_parquet('{TEST_NORMALIZED_FILES["s1"].as_posix()}')
    ) TO '{S1_BUCKET_DIR.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY(candidate_bucket), COMPRESSION ZSTD)
""")


def append_query_rows(connection, query, writer, expected_columns):
    reader = connection.execute(query).fetch_record_batch(rows_per_batch=25000)
    written = 0
    for batch in reader:
        values = batch.to_pydict()
        writer.writerows(zip(*(values[column] for column in expected_columns)))
        written += batch.num_rows
    return written


for path in [MATCHING_RESULTS_FILE, CANDIDATE_PAIRS_FILE]:
    if path.exists():
        path.unlink()

matching_rows = 0
candidate_rows = 0
with open(MATCHING_RESULTS_FILE, "w", encoding="utf-8", newline="") as matching_stream, \
     open(CANDIDATE_PAIRS_FILE, "w", encoding="utf-8", newline="") as candidate_stream:
    matching_writer = csv.writer(
        matching_stream, delimiter="\t", lineterminator="\n",
        quoting=csv.QUOTE_NONE, escapechar="\\",
    )
    candidate_writer = csv.writer(
        candidate_stream, delimiter="\t", lineterminator="\n",
        quoting=csv.QUOTE_NONE, escapechar="\\",
    )
    matching_writer.writerow(["source1_entity_id", "matched_entity_ids"])
    candidate_writer.writerow(["source1_entity_id", "candidate_entity_ids"])

    progress = tqdm(range(CANDIDATE_BUCKETS), desc="Final TSV buckets")
    for bucket in progress:
        s1_pattern = S1_BUCKET_DIR / f"candidate_bucket={bucket}" / "*.parquet"
        candidate_file = TEST_CANDIDATE_SHARDS_LOCAL / f"test_candidates_{bucket:03d}.parquet"
        selected_file = TEST_SELECTED_LOCAL / f"test_selected_{bucket:03d}.parquet"

        matching_query = f"""
            WITH grouped AS (
                SELECT source1_entity_id,
                       string_agg(candidate_entity_id, ',') AS matched_entity_ids
                FROM read_parquet('{selected_file.as_posix()}')
                GROUP BY source1_entity_id
            )
            SELECT s1.entity_id AS source1_entity_id,
                   COALESCE(grouped.matched_entity_ids, '') AS matched_entity_ids
            FROM read_parquet('{s1_pattern.as_posix()}') s1
            LEFT JOIN grouped ON s1.entity_id=grouped.source1_entity_id
        """
        candidate_query = f"""
            WITH grouped AS (
                SELECT source1_entity_id,
                       string_agg(candidate_entity_id, ',') AS candidate_entity_ids
                FROM read_parquet('{candidate_file.as_posix()}')
                GROUP BY source1_entity_id
            )
            SELECT s1.entity_id AS source1_entity_id,
                   COALESCE(grouped.candidate_entity_ids, '') AS candidate_entity_ids
            FROM read_parquet('{s1_pattern.as_posix()}') s1
            LEFT JOIN grouped ON s1.entity_id=grouped.source1_entity_id
        """
        matching_rows += append_query_rows(
            connection, matching_query, matching_writer,
            ["source1_entity_id", "matched_entity_ids"],
        )
        candidate_rows += append_query_rows(
            connection, candidate_query, candidate_writer,
            ["source1_entity_id", "candidate_entity_ids"],
        )
        progress.set_postfix(matching=f"{matching_rows:,}", candidates=f"{candidate_rows:,}")

connection.close()
if matching_rows != test_s1_count or candidate_rows != test_s1_count:
    raise RuntimeError(
        f"Row-count mismatch: expected={test_s1_count:,}, "
        f"matching={matching_rows:,}, candidates={candidate_rows:,}"
    )
print("matching_results.tsv:", MATCHING_RESULTS_FILE, MATCHING_RESULTS_FILE.stat().st_size / 2**20, "MB")
print("candidate_pairs.tsv:", CANDIDATE_PAIRS_FILE, CANDIDATE_PAIRS_FILE.stat().st_size / 2**20, "MB")


Final TSV buckets:   0%|          | 0/32 [00:00<?, ?it/s]

/tmp/ipykernel_927/2452170980.py:22: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  reader = connection.execute(query).fetch_record_batch(rows_per_batch=25000)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

matching_results.tsv: /content/amlc_second_try/test/output/matching_results.tsv 92.5955457687378 MB
candidate_pairs.tsv: /content/amlc_second_try/test/output/candidate_pairs.tsv 2846.7252340316772 MB


## Block 27 — Validate and save final submission files under Second Try/outputs


In [15]:
validator_matches = list(WORK_DIR.rglob("validate_submission.py"))
if not validator_matches:
    raise FileNotFoundError("validate_submission.py was not found in the resource archive.")
validator = validator_matches[0]
validation_result = subprocess.run(
    [
        sys.executable, str(validator),
        "--matching", str(MATCHING_RESULTS_FILE),
        "--test-dir", str(TEST_DIR),
    ],
    capture_output=True,
    text=True,
)
print(validation_result.stdout)
if validation_result.stderr:
    print(validation_result.stderr)
if validation_result.returncode != 0:
    raise RuntimeError("Submission validation failed. Do not upload the files.")

print(
    "The official validator was intentionally run on matching_results.tsv only. "
    "Loading the much larger enhanced candidate file into its Python set mapping "
    "can exceed Colab RAM. Block 26 already verified exactly one candidate row "
    "for every test S1, and all candidate IDs came directly from S2/S3."
)

DRIVE_MATCHING_RESULTS = PERSIST_OUTPUTS / "matching_results.tsv"
DRIVE_CANDIDATE_PAIRS = PERSIST_OUTPUTS / "candidate_pairs.tsv"
atomic_copy(MATCHING_RESULTS_FILE, DRIVE_MATCHING_RESULTS)
atomic_copy(CANDIDATE_PAIRS_FILE, DRIVE_CANDIDATE_PAIRS)

run_manifest = {
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "validation_macro_f0_5": saved_metrics["best_validation_result"]["macro_f0_5"],
    "threshold_s2": TEST_THRESHOLD_S2,
    "threshold_s3": TEST_THRESHOLD_S3,
    "test_s1_rows": test_s1_count,
    "test_candidate_pairs": test_candidate_count,
    "matching_results": str(DRIVE_MATCHING_RESULTS),
    "candidate_pairs": str(DRIVE_CANDIDATE_PAIRS),
}
manifest_path = PERSIST_OUTPUTS / "run_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as stream:
    json.dump(run_manifest, stream, indent=2)

print("Validation passed.")
print("UPLOAD THIS FILE:", DRIVE_MATCHING_RESULTS)
print("Candidate artifact:", DRIVE_CANDIDATE_PAIRS)
print("Manifest:", manifest_path)


ML Challenge 2026 — submission validator
  test dir: /content/amlc_second_try_resource/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (119298 empty, 1613246 non-empty).

PASS — no blocking issues found. Safe to submit.

The official validator was intentionally run on matching_results.tsv only. Loading the much larger enhanced candidate file into its Python set mapping can exceed Colab RAM. Block 26 already verified exactly one candidate row for every test S1, and all candidate IDs came directly from S2/S3.
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/outputs/matching_results.tsv | size GB: 0.09 | minutes: 0.01
Checkpointed: /content/drive/MyDrive/AMLC Second Try Shared/outputs/candidate_pairs.tsv | size GB: 2.78 | minutes: 0.43
Validation passed.
UPLOAD THIS FILE: /content/drive/MyDrive/AMLC Second Try Shared/outputs/matching_results.tsv
Candidate artifact: /content/drive/MyDrive/AMLC Second Try Shared/outputs/candidate

**Final generation block**

In [ ]:
# ============================================================
# COUNTRY-AWARE THRESHOLD + COUNTRY-AWARE CONFLICT MARGIN
#
# US/India threshold: 0.37
# France threshold: 0.85
#
# US/India uniqueness margin: 0.02
# France uniqueness margin: 0.10
#
# CPU is sufficient.
# ============================================================

from pathlib import Path
import csv
import shutil

import duckdb
from tqdm.auto import tqdm
from IPython.display import display


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/AMLC Second Try Shared"
)

SELECTED_DIRECTORY = (
    ROOT
    / "testing"
    / "selected_shards"
)

SELECTED_PATTERN = (
    SELECTED_DIRECTORY
    / "test_selected_*.parquet"
)

TEST_S1_FILE = (
    ROOT
    / "testing"
    / "normalized"
    / "test_s1.parquet"
)

DEFAULT_THRESHOLD = 0.37
FRANCE_THRESHOLD = 0.85

DEFAULT_MARGIN = 0.02
FRANCE_MARGIN = 0.10

LOCAL_OUTPUT = Path(
    "/content/matching_results_france_085_margin_010.tsv"
)

DRIVE_OUTPUT = (
    ROOT
    / "outputs"
    / "matching_results_france_085_margin_010.tsv"
)


# ------------------------------------------------------------
# 2. Verify inputs
# ------------------------------------------------------------

selected_files = sorted(
    SELECTED_DIRECTORY.glob(
        "test_selected_*.parquet"
    )
)

print(
    "Selected test shards:",
    len(selected_files),
)

if len(selected_files) != 32:
    raise RuntimeError(
        "Expected 32 selected test shards, "
        f"but found {len(selected_files)}."
    )

if not TEST_S1_FILE.exists():
    raise FileNotFoundError(
        TEST_S1_FILE
    )

DRIVE_OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if LOCAL_OUTPUT.exists():
    LOCAL_OUTPUT.unlink()


# ------------------------------------------------------------
# 3. Configure DuckDB
# ------------------------------------------------------------

TEMP_DIRECTORY = Path(
    "/content/france_margin_010_temp"
)

TEMP_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

DATABASE_FILE = Path(
    "/content/france_margin_010.duckdb"
)

if DATABASE_FILE.exists():
    DATABASE_FILE.unlink()

connection = duckdb.connect(
    DATABASE_FILE.as_posix()
)

connection.execute(
    "SET threads=2"
)

connection.execute(
    "SET memory_limit='5GB'"
)

connection.execute(
    "SET preserve_insertion_order=false"
)

connection.execute(
    f"SET temp_directory='{TEMP_DIRECTORY.as_posix()}'"
)


# ------------------------------------------------------------
# 4. Count expected output rows
# ------------------------------------------------------------

expected_rows = connection.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{TEST_S1_FILE.as_posix()}'
)
""").fetchone()[0]

print(
    "Expected S1 rows:",
    f"{expected_rows:,}",
)


# ------------------------------------------------------------
# 5. Display country counts
# ------------------------------------------------------------

country_counts = connection.execute(f"""
SELECT
    LOWER(TRIM(country_norm)) AS country,
    COUNT(*) AS s1_records

FROM read_parquet(
    '{TEST_S1_FILE.as_posix()}'
)

GROUP BY 1
ORDER BY 1
""").fetch_df()

print("\nTest S1 records by country:")
display(country_counts)


# ------------------------------------------------------------
# 6. Build country-aware post-processing query
# ------------------------------------------------------------

query = f"""
WITH selected_with_country AS (
    SELECT DISTINCT
        selected.source1_entity_id,
        selected.candidate_entity_id,
        selected.match_probability,

        LOWER(
            TRIM(s1.country_norm)
        ) AS country

    FROM read_parquet(
        '{SELECTED_PATTERN.as_posix()}'
    ) AS selected

    INNER JOIN read_parquet(
        '{TEST_S1_FILE.as_posix()}'
    ) AS s1

    ON selected.source1_entity_id
       = s1.entity_id
),

threshold_filtered AS (
    SELECT *

    FROM selected_with_country

    WHERE
        (
            country = 'france'

            AND match_probability
                >= {FRANCE_THRESHOLD}
        )

        OR

        (
            country <> 'france'

            AND match_probability
                >= {DEFAULT_THRESHOLD}
        )
),

ranked AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY candidate_entity_id

            ORDER BY
                match_probability DESC,
                source1_entity_id
        ) AS target_rank,

        LEAD(match_probability) OVER (
            PARTITION BY candidate_entity_id

            ORDER BY
                match_probability DESC,
                source1_entity_id
        ) AS second_probability

    FROM threshold_filtered
),

retained AS (
    SELECT
        source1_entity_id,
        candidate_entity_id,
        match_probability,
        country

    FROM ranked

    WHERE
        target_rank = 1

        AND (
            second_probability IS NULL

            OR (
                country = 'france'

                AND (
                    match_probability
                    - second_probability
                ) >= {FRANCE_MARGIN}
            )

            OR (
                country <> 'france'

                AND (
                    match_probability
                    - second_probability
                ) >= {DEFAULT_MARGIN}
            )
        )
),

grouped AS (
    SELECT
        source1_entity_id,

        string_agg(
            candidate_entity_id,
            ','
            ORDER BY
                match_probability DESC,
                candidate_entity_id
        ) AS matched_entity_ids

    FROM retained

    GROUP BY source1_entity_id
)

SELECT
    s1.entity_id AS source1_entity_id,

    COALESCE(
        grouped.matched_entity_ids,
        ''
    ) AS matched_entity_ids

FROM read_parquet(
    '{TEST_S1_FILE.as_posix()}'
) AS s1

LEFT JOIN grouped
ON s1.entity_id = grouped.source1_entity_id

ORDER BY s1.entity_id
"""


# ------------------------------------------------------------
# 7. Print applied configuration
# ------------------------------------------------------------

print("\nConfiguration:")
print(
    "US/India threshold:",
    DEFAULT_THRESHOLD,
)
print(
    "France threshold:",
    FRANCE_THRESHOLD,
)
print(
    "US/India margin:",
    DEFAULT_MARGIN,
)
print(
    "France margin:",
    FRANCE_MARGIN,
)


# ------------------------------------------------------------
# 8. Run query and stream results
# ------------------------------------------------------------

print(
    "\nApplying country-aware filtering..."
)

reader = (
    connection
    .execute(query)
    .fetch_record_batch(
        rows_per_batch=50000
    )
)


# ------------------------------------------------------------
# 9. Write TSV
# ------------------------------------------------------------

written_rows = 0

with open(
    LOCAL_OUTPUT,
    "w",
    encoding="utf-8",
    newline="",
) as stream:

    writer = csv.writer(
        stream,
        delimiter="\t",
        lineterminator="\n",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
    )

    writer.writerow([
        "source1_entity_id",
        "matched_entity_ids",
    ])

    progress = tqdm(
        total=expected_rows,
        desc="Writing France-margin submission",
        unit="rows",
    )

    for batch in reader:
        values = batch.to_pydict()

        writer.writerows(
            zip(
                values["source1_entity_id"],
                values["matched_entity_ids"],
            )
        )

        written_rows += batch.num_rows
        progress.update(batch.num_rows)

        progress.set_postfix(
            written=f"{written_rows:,}"
        )

    progress.close()

connection.close()


# ------------------------------------------------------------
# 10. Validate output
# ------------------------------------------------------------

if written_rows != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows:,} rows, "
        f"but wrote {written_rows:,}."
    )

if not LOCAL_OUTPUT.exists():
    raise RuntimeError(
        "Output file was not created."
    )

output_size_mb = (
    LOCAL_OUTPUT.stat().st_size
    / 2**20
)

if output_size_mb <= 0:
    raise RuntimeError(
        "Generated output is empty."
    )


# ------------------------------------------------------------
# 11. Copy result to Google Drive
# ------------------------------------------------------------

shutil.copy2(
    LOCAL_OUTPUT,
    DRIVE_OUTPUT,
)

print(
    "\nCountry-aware margin submission completed."
)

print(
    "Rows written:",
    f"{written_rows:,}",
)

print(
    "Local output:",
    LOCAL_OUTPUT,
)

print(
    "Drive output:",
    DRIVE_OUTPUT,
)

print(
    "File size:",
    round(output_size_mb, 2),
    "MB",
)

print(
    "\nSubmit this matching file with the existing "
    "candidate_pairs.tsv."
)